# Neural Feature Explorer & Decoder Lab

**Data:** immutable, fold-independent DEV feature cache only; no raw
neural bundle and no CONFIRM access.

[Release provenance](https://github.com/c-lin-chunyi/nma-project-data-analysis/releases/tag/neural-dev-features-v1-29482249873)

Run the cells from top to bottom. This notebook is self-contained: it
does not clone a repository. Its first code cell installs only missing
dependencies. The complete feature cache is about 24.3 MiB compressed.


## 0. Prepare the Colab runtime


In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version
from importlib.util import find_spec

requirements = [
    ("pandas", "pandas", "pandas"),
    ("pyarrow", "pyarrow", "pyarrow"),
    ("h5py", "h5py", "h5py"),
    ("sklearn", "scikit-learn", "scikit-learn"),
    ("matplotlib", "matplotlib", "matplotlib"),
    ("seaborn", "seaborn", "seaborn"),
]
install = []
for module, package, distribution in requirements:
    if find_spec(module) is None:
        install.append(package)

if install:
    print("Installing missing Colab dependencies:", ", ".join(install))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *install
    ])
else:
    print("All notebook dependencies are already available.")

versions = {}
for _, _, distribution in requirements:
    try:
        versions[distribution] = version(distribution)
    except PackageNotFoundError:
        versions[distribution] = "installed in this cell; restart only if import fails"
print("Runtime versions:", versions)


## 1. Verified release loader

Embedded for one-file Colab use.


In [ ]:
"""Verified, cache-aware readers for the immutable public DEV releases.

The notebook helpers intentionally avoid AllenSDK and the 16 GB neural bundle.
They consume only the compact behavioral scan and analysis-ready feature cache.
"""

from __future__ import annotations

import hashlib
import json
import os
import shutil
import sys
import tarfile
import urllib.error
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Mapping

import h5py
import numpy as np
import pandas as pd

REPOSITORY = "c-lin-chunyi/nma-project-data-analysis"
BEHAVIORAL_TAG = "behavioral-v3.1-29482141350"
FEATURE_TAG = "neural-dev-features-v1-29482249873"
BEHAVIORAL_ARCHIVE = "behavioral-v3.1-scan.tar.gz"
FEATURE_MANIFEST = "feature-cache-manifest.json"
FEATURE_NAMES = (
    "events_baselined_post",
    "events_unbaselined_pre",
    "events_unbaselined_post",
    "events_baselined_full_pre",
    "dff_baselined_post",
)

BEHAVIORAL_REQUIRED_COLUMNS = {
    "trial_labels": {
        "behavior_session_id",
        "mouse_id",
        "trial_id",
        "trial_index",
        "late_hit",
        "early_hit",
        "miss",
        "aborted",
        "engaged_A",
        "engaged_B",
        "engaged_A_hysteretic",
        "keep_A",
        "keep_B",
        "keep_A_hysteretic",
    },
    "session_scan": {
        "behavior_session_id",
        "mouse_id",
        "n_trials",
        "late_hit_B",
        "miss_B",
        "behavioral_eligible",
    },
}


class ReleaseDataError(RuntimeError):
    """Raised when a release asset is unavailable, unsafe, or invalid."""


def default_cache_dir() -> Path:
    override = os.environ.get("NMA_RELEASE_CACHE")
    if override:
        return Path(override).expanduser()
    if "google.colab" in sys.modules or Path("/content").is_dir():
        return Path("/content/nma-release-cache")
    return Path.home() / ".cache" / "nma-project-data-analysis"


def release_url(tag: str, asset: str) -> str:
    return f"https://github.com/{REPOSITORY}/releases/download/{tag}/{asset}"


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while chunk := stream.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def parse_sha256sums(path: Path) -> dict[str, str]:
    result: dict[str, str] = {}
    for line in path.read_text().splitlines():
        parts = line.strip().split()
        if len(parts) >= 2 and len(parts[0]) == 64:
            result[parts[-1].lstrip("*")] = parts[0].lower()
    if not result:
        raise ReleaseDataError(f"No SHA-256 entries found in {path}")
    return result


def _progress(name: str, block: int, block_size: int, total: int) -> None:
    if total <= 0:
        return
    downloaded = min(block * block_size, total)
    pct = 100 * downloaded / total
    print(f"\r{name}: {downloaded / 2**20:.1f}/{total / 2**20:.1f} MiB ({pct:5.1f}%)",
          end="", flush=True)
    if downloaded >= total:
        print()


def _copy_or_download(
    tag: str,
    asset: str,
    destination: Path,
    *,
    expected_sha256: str | None = None,
    source_dir: Path | None = None,
    show_progress: bool = True,
) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file() and (
        expected_sha256 is None or sha256_file(destination) == expected_sha256.lower()
    ):
        return destination
    if destination.exists():
        destination.unlink()

    partial = destination.with_name(destination.name + ".part")
    if partial.exists():
        partial.unlink()
    try:
        if source_dir is not None:
            source = Path(source_dir) / asset
            if not source.is_file():
                raise ReleaseDataError(f"Missing local release asset: {source}")
            shutil.copyfile(source, partial)
        else:
            callback = (
                (lambda block, size, total: _progress(asset, block, size, total))
                if show_progress else None
            )
            urllib.request.urlretrieve(release_url(tag, asset), partial, callback)
        if expected_sha256 is not None:
            actual = sha256_file(partial)
            if actual != expected_sha256.lower():
                raise ReleaseDataError(
                    f"SHA-256 mismatch for {asset}: expected {expected_sha256}, got {actual}"
                )
        os.replace(partial, destination)
    except (OSError, urllib.error.URLError) as exc:
        if partial.exists():
            partial.unlink()
        if isinstance(exc, ReleaseDataError):
            raise
        raise ReleaseDataError(f"Could not obtain {asset}: {exc}") from exc
    except Exception:
        if partial.exists():
            partial.unlink()
        raise
    return destination


def _safe_extract_tar(archive: Path, destination: Path) -> None:
    """Extract regular files/directories without trusting archive paths or links."""
    destination.mkdir(parents=True, exist_ok=True)
    base = destination.resolve()
    with tarfile.open(archive, "r:*") as tar:
        for member in tar:
            relative = Path(member.name)
            if relative.is_absolute() or ".." in relative.parts:
                raise ReleaseDataError(f"Unsafe path in {archive.name}: {member.name}")
            target = (destination / relative).resolve()
            try:
                target.relative_to(base)
            except ValueError as exc:
                raise ReleaseDataError(
                    f"Archive member escapes destination: {member.name}"
                ) from exc
            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            if not member.isfile():
                raise ReleaseDataError(
                    f"Unsupported link/device in {archive.name}: {member.name}"
                )
            source = tar.extractfile(member)
            if source is None:
                raise ReleaseDataError(f"Could not read {member.name} from {archive.name}")
            target.parent.mkdir(parents=True, exist_ok=True)
            with source, target.open("wb") as output:
                shutil.copyfileobj(source, output)


def _extract_once(archive: Path, destination: Path, expected_sha256: str) -> None:
    marker = destination / ".archive-sha256"
    if marker.is_file() and marker.read_text().strip() == expected_sha256:
        return
    if destination.exists():
        shutil.rmtree(destination)
    _safe_extract_tar(archive, destination)
    marker.write_text(expected_sha256 + "\n")


@dataclass(frozen=True)
class BehaviorScan:
    tag: str
    root: Path
    tables: Mapping[str, pd.DataFrame]
    manifest: Mapping[str, Any]

    def __getitem__(self, name: str) -> pd.DataFrame:
        return self.tables[name]

    @property
    def trial_labels(self) -> pd.DataFrame:
        return self.tables["trial_labels"]

    @property
    def session_scan(self) -> pd.DataFrame:
        return self.tables["session_scan"]


def _behavior_table_name(path: Path) -> str:
    return path.stem.lstrip("_")


def _validate_behavior(scan: BehaviorScan, *, strict_public: bool) -> None:
    if scan.manifest.get("schema") != "behavioral-v3.1":
        raise ReleaseDataError("Unexpected behavioral manifest schema")
    for name, required in BEHAVIORAL_REQUIRED_COLUMNS.items():
        if name not in scan.tables:
            raise ReleaseDataError(f"Behavioral scan is missing {name}.parquet")
        missing = required - set(scan.tables[name].columns)
        if missing:
            raise ReleaseDataError(f"{name}.parquet is missing columns: {sorted(missing)}")
    if strict_public:
        if int(scan.manifest.get("n_dev_sessions", -1)) != 50:
            raise ReleaseDataError("Public behavioral release must contain 50 DEV sessions")
        if scan.session_scan["behavior_session_id"].nunique() != 50:
            raise ReleaseDataError("Behavioral session table does not contain 50 sessions")


def load_behavioral_scan(
    cache_dir: str | Path | None = None,
    *,
    source_dir: str | Path | None = None,
    show_progress: bool = True,
) -> BehaviorScan:
    """Download, verify, extract, and read the compact behavioral DEV scan."""
    cache = Path(cache_dir) if cache_dir is not None else default_cache_dir()
    source_value = source_dir or os.environ.get("NMA_BEHAVIORAL_SOURCE_DIR")
    local_source = Path(source_value) if source_value is not None else None
    root = cache / "behavioral" / BEHAVIORAL_TAG
    sums_path = _copy_or_download(
        BEHAVIORAL_TAG, "SHA256SUMS", root / "SHA256SUMS",
        source_dir=local_source, show_progress=show_progress,
    )
    sums = parse_sha256sums(sums_path)
    expected = sums.get(BEHAVIORAL_ARCHIVE)
    if expected is None:
        raise ReleaseDataError(f"SHA256SUMS does not cover {BEHAVIORAL_ARCHIVE}")
    archive = _copy_or_download(
        BEHAVIORAL_TAG, BEHAVIORAL_ARCHIVE, root / BEHAVIORAL_ARCHIVE,
        expected_sha256=expected, source_dir=local_source, show_progress=show_progress,
    )
    manifest_sha = sums.get("behavioral-manifest.json")
    manifest_path = _copy_or_download(
        BEHAVIORAL_TAG, "behavioral-manifest.json", root / "behavioral-manifest.json",
        expected_sha256=manifest_sha, source_dir=local_source, show_progress=show_progress,
    )
    extracted = root / "scan"
    _extract_once(archive, extracted, expected)
    try:
        tables = {
            _behavior_table_name(path): pd.read_parquet(path)
            for path in sorted(extracted.glob("*.parquet"))
        }
        manifest = json.loads(manifest_path.read_text())
    except Exception as exc:
        raise ReleaseDataError(f"Could not read behavioral scan files: {exc}") from exc
    scan = BehaviorScan(BEHAVIORAL_TAG, extracted, tables, manifest)
    _validate_behavior(scan, strict_public=local_source is None)
    return scan


@dataclass(frozen=True)
class FeatureMatrix:
    values: np.ndarray
    trial_ids: np.ndarray
    cell_ids: np.ndarray
    name: str
    experiment_id: int


class FeatureCache:
    """Lazy reader over the extracted, experiment-level feature cache."""

    feature_names = FEATURE_NAMES

    def __init__(
        self,
        root: Path,
        manifest: Mapping[str, Any],
        validation: Mapping[str, Any],
        experiments: pd.DataFrame,
        *,
        strict_public: bool = True,
    ) -> None:
        self.root = Path(root)
        self.manifest = dict(manifest)
        self.validation = dict(validation)
        self.source_experiments = experiments.copy()
        meta_rows = []
        self._meta: dict[int, dict[str, Any]] = {}
        for path in sorted(self.root.glob("*.feature-meta.json")):
            meta = json.loads(path.read_text())
            identity = dict(meta["identity"])
            oeid = int(identity["ophys_experiment_id"])
            self._meta[oeid] = meta
            meta_rows.append({
                **identity,
                "n_trials": int(meta["n_trials"]),
                "n_cells": int(meta["n_cells"]),
            })
        self.index = pd.DataFrame(meta_rows).sort_values(
            ["mouse_id", "ophys_container_id", "ophys_experiment_id"]
        ).reset_index(drop=True)
        self._validate(strict_public=strict_public)

    @property
    def experiment_ids(self) -> list[int]:
        return self.index["ophys_experiment_id"].astype(int).tolist()

    def meta(self, experiment_id: int) -> dict[str, Any]:
        try:
            return self._meta[int(experiment_id)]
        except KeyError as exc:
            raise KeyError(f"Unknown feature-cache experiment: {experiment_id}") from exc

    def labels(self, experiment_id: int) -> pd.DataFrame:
        path = self.root / f"{int(experiment_id)}.labels.parquet"
        return pd.read_parquet(path)

    def q2(self, experiment_id: int) -> pd.DataFrame:
        path = self.root / f"{int(experiment_id)}.q2.parquet"
        return pd.read_parquet(path)

    def matrix(self, experiment_id: int, name: str = FEATURE_NAMES[0]) -> FeatureMatrix:
        if name not in FEATURE_NAMES:
            raise KeyError(f"Unknown feature {name!r}; choose one of {FEATURE_NAMES}")
        oeid = int(experiment_id)
        with h5py.File(self.root / f"{oeid}.features.h5", "r") as h5:
            values = h5[name][:]
            trial_ids = h5["trial_id"][:]
            cell_ids = h5["cell_specimen_id"][:]
        return FeatureMatrix(values, trial_ids, cell_ids, name, oeid)

    def _validate(self, *, strict_public: bool) -> None:
        if self.manifest.get("schema") != "neural-dev-feature-cache-v1":
            raise ReleaseDataError("Unexpected feature-cache manifest schema")
        if not bool(self.validation.get("complete")):
            raise ReleaseDataError(
                f"Feature-cache validation is not complete: {self.validation.get('failures')}"
            )
        if self.index.empty:
            raise ReleaseDataError("No feature experiments were extracted")
        expected = int(self.manifest.get("n_active_experiments", -1))
        if len(self.index) != expected:
            raise ReleaseDataError(
                f"Expected {expected} active experiments, found {len(self.index)}"
            )
        if strict_public and (
            expected != 50 or int(self.manifest.get("n_containers", -1)) != 10
        ):
            raise ReleaseDataError("Public feature cache must contain 50 experiments/10 containers")
        for oeid in self.experiment_ids:
            files = [
                self.root / f"{oeid}.features.h5",
                self.root / f"{oeid}.labels.parquet",
                self.root / f"{oeid}.q2.parquet",
            ]
            if not all(path.is_file() for path in files):
                missing = [path.name for path in files if not path.is_file()]
                raise ReleaseDataError(f"Experiment {oeid} is missing files: {missing}")
            labels = self.labels(oeid)
            q2 = self.q2(oeid)
            with h5py.File(files[0], "r") as h5:
                missing_features = set(FEATURE_NAMES) - set(h5.keys())
                if missing_features:
                    raise ReleaseDataError(
                        f"Experiment {oeid} is missing HDF5 datasets: {sorted(missing_features)}"
                    )
                trial_ids = h5["trial_id"][:]
                n_cells = len(h5["cell_specimen_id"])
                for feature in FEATURE_NAMES:
                    if h5[feature].shape != (len(trial_ids), n_cells):
                        raise ReleaseDataError(
                            f"Experiment {oeid} has invalid shape for {feature}"
                        )
            if not np.array_equal(labels["trial_id"].to_numpy(), trial_ids):
                raise ReleaseDataError(f"Experiment {oeid} label/HDF5 trial IDs do not align")
            if not np.array_equal(q2["trial_id"].to_numpy(), trial_ids):
                raise ReleaseDataError(f"Experiment {oeid} Q2/HDF5 trial IDs do not align")


def load_feature_cache(
    cache_dir: str | Path | None = None,
    *,
    source_dir: str | Path | None = None,
    show_progress: bool = True,
) -> FeatureCache:
    """Download and verify all compact feature shards, then return a lazy reader."""
    cache = Path(cache_dir) if cache_dir is not None else default_cache_dir()
    source_value = source_dir or os.environ.get("NMA_FEATURE_SOURCE_DIR")
    local_source = Path(source_value) if source_value is not None else None
    root = cache / "features" / FEATURE_TAG
    sums_path = _copy_or_download(
        FEATURE_TAG, "SHA256SUMS", root / "SHA256SUMS",
        source_dir=local_source, show_progress=show_progress,
    )
    sums = parse_sha256sums(sums_path)
    metadata_names = [
        FEATURE_MANIFEST,
        "feature-cache-validation.json",
        "dev_experiments.csv",
    ]
    metadata: dict[str, Path] = {}
    for name in metadata_names:
        expected = sums.get(name)
        if expected is None:
            raise ReleaseDataError(f"SHA256SUMS does not cover {name}")
        metadata[name] = _copy_or_download(
            FEATURE_TAG, name, root / name, expected_sha256=expected,
            source_dir=local_source, show_progress=show_progress,
        )
    manifest = json.loads(metadata[FEATURE_MANIFEST].read_text())
    validation = json.loads(metadata["feature-cache-validation.json"].read_text())
    extracted = root / "cache"
    parts = manifest.get("parts", [])
    if not parts:
        raise ReleaseDataError("Feature manifest lists no shards")
    for part in parts:
        name = str(part["name"])
        expected = str(part["sha256"]).lower()
        archive = _copy_or_download(
            FEATURE_TAG, name, root / "archives" / name,
            expected_sha256=expected, source_dir=local_source, show_progress=show_progress,
        )
        marker = extracted / f".{name}.sha256"
        if not marker.is_file() or marker.read_text().strip() != expected:
            _safe_extract_tar(archive, extracted)
            marker.parent.mkdir(parents=True, exist_ok=True)
            marker.write_text(expected + "\n")
    experiments = pd.read_csv(metadata["dev_experiments.csv"])
    try:
        return FeatureCache(
            extracted,
            manifest,
            validation,
            experiments,
            strict_public=local_source is None,
        )
    except ReleaseDataError:
        raise
    except Exception as exc:
        raise ReleaseDataError(f"Could not read feature-cache files: {exc}") from exc


## 2. Typed single-experiment decoder

Embedded for one-file Colab use.


In [ ]:
"""A typed, single-experiment Q1 decoder for the educational Colab notebook."""

from __future__ import annotations

import hashlib
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler



@dataclass(frozen=True)
class DecoderConfig:
    k: int | None = 50
    C: float = 1e-4
    n_seeds: int = 10
    n_folds: int = 5
    purge: int = 10
    cv: str = "blocked"

    @property
    def exploratory(self) -> bool:
        return not (
            self.k == 50
            and self.C == 1e-4
            and self.n_seeds == 10
            and self.n_folds == 5
            and self.purge == 10
            and self.cv == "blocked"
        )

    def validate(self) -> None:
        if self.k is not None and self.k < 1:
            raise ValueError("k must be positive or None for all cells")
        if self.C <= 0:
            raise ValueError("C must be positive")
        if self.n_seeds < 1:
            raise ValueError("n_seeds must be positive")
        if self.n_folds < 2:
            raise ValueError("n_folds must be at least 2")
        if self.purge < 0:
            raise ValueError("purge must be non-negative")
        if self.cv not in {"blocked", "random"}:
            raise ValueError("cv must be 'blocked' or 'random'")


@dataclass
class DecoderResult:
    status: str
    reason: str | None
    config: DecoderConfig
    experiment_id: int
    feature_name: str
    seed_metrics: pd.DataFrame = field(default_factory=pd.DataFrame)
    fold_metrics: pd.DataFrame = field(default_factory=pd.DataFrame)
    oof: pd.DataFrame = field(default_factory=pd.DataFrame)
    cell_summary: pd.DataFrame = field(default_factory=pd.DataFrame)

    @property
    def mean_auc(self) -> float:
        if self.seed_metrics.empty:
            return float("nan")
        return float(self.seed_metrics["auc"].mean())


def contiguous_purged_folds(
    raw_index: np.ndarray, n_blocks: int = 5, purge: int = 10
) -> list[tuple[np.ndarray, np.ndarray]]:
    raw_index = np.asarray(raw_index, dtype=int)
    order = np.argsort(raw_index)
    result = []
    for test in np.array_split(order, n_blocks):
        if not len(test):
            continue
        low, high = int(raw_index[test].min()), int(raw_index[test].max())
        train = np.setdiff1d(order, test)
        adjacent = (
            ((raw_index[train] >= low - purge) & (raw_index[train] < low))
            | ((raw_index[train] > high) & (raw_index[train] <= high + purge))
        )
        result.append((train[~adjacent], test))
    return result


def deterministic_cell_indices(
    n_cells: int, k: int | None, seed: int, experiment_id: int
) -> np.ndarray | None:
    if k is None:
        return np.arange(n_cells)
    if n_cells < k:
        return None
    digest = hashlib.sha256(f"{int(experiment_id)}:{int(k)}:{int(seed)}".encode()).digest()
    rng = np.random.default_rng(int.from_bytes(digest[:8], "big"))
    return np.sort(rng.choice(n_cells, int(k), replace=False))


def _empty_result(
    reason: str,
    config: DecoderConfig,
    matrix: FeatureMatrix,
) -> DecoderResult:
    return DecoderResult(
        status="nonestimable",
        reason=reason,
        config=config,
        experiment_id=matrix.experiment_id,
        feature_name=matrix.name,
    )


def run_q1_decoder(
    matrix: FeatureMatrix,
    labels: pd.DataFrame,
    config: DecoderConfig | None = None,
) -> DecoderResult:
    """Fit the registered single-session late-hit-vs-miss decoder.

    This follows the v3.2/v3.3 Q1 session defaults but does not perform the
    authoritative mouse-level aggregation.
    """
    config = config or DecoderConfig()
    config.validate()
    required = {
        "trial_id", "trial_index", "engaged_B", "keep_B", "late_hit", "miss",
    }
    missing = required - set(labels.columns)
    if missing:
        return _empty_result(f"missing_label_columns:{','.join(sorted(missing))}", config, matrix)
    aligned = labels.set_index("trial_id").reindex(matrix.trial_ids).reset_index()
    if aligned["trial_index"].isna().any():
        return _empty_result("trial_alignment_failed", config, matrix)
    mask = (
        aligned["engaged_B"].fillna(False).astype(bool)
        & aligned["keep_B"].fillna(False).astype(bool)
        & (
            aligned["late_hit"].fillna(False).astype(bool)
            | aligned["miss"].fillna(False).astype(bool)
        )
    )
    X = np.asarray(matrix.values[mask.to_numpy()], dtype=float)
    selected_labels = aligned.loc[mask].reset_index(drop=True)
    y = selected_labels["late_hit"].astype(int).to_numpy()
    raw = selected_labels["trial_index"].astype(int).to_numpy()
    if len(np.unique(y)) < 2:
        return _empty_result("insufficient_classes", config, matrix)
    if not np.isfinite(X).all():
        return _empty_result("nonfinite_features", config, matrix)
    if config.k is not None and X.shape[1] < config.k:
        return _empty_result("low_cells", config, matrix)

    seed_rows: list[dict] = []
    fold_rows: list[dict] = []
    coefficient_rows: list[dict] = []
    oof_columns: dict[str, np.ndarray] = {}
    selected_by_seed: dict[int, np.ndarray] = {}

    for seed in range(config.n_seeds):
        cell_index = deterministic_cell_indices(
            X.shape[1], config.k, seed, matrix.experiment_id
        )
        if cell_index is None:
            return _empty_result("low_cells", config, matrix)
        selected_by_seed[seed] = cell_index
        X_seed = X[:, cell_index]
        scores = np.full(len(y), np.nan)
        if config.cv == "blocked":
            splits = contiguous_purged_folds(raw, config.n_folds, config.purge)
        else:
            class_counts = np.bincount(y, minlength=2)
            if int(class_counts.min()) < config.n_folds:
                return _empty_result("random_cv_class_support_nonestimable", config, matrix)
            splits = list(
                StratifiedKFold(
                    config.n_folds, shuffle=True, random_state=seed
                ).split(X_seed, y)
            )
        if len(splits) != config.n_folds:
            return _empty_result("fold_construction_failed", config, matrix)

        for fold, (train, test) in enumerate(splits):
            train_classes = np.bincount(y[train], minlength=2)
            test_classes = np.bincount(y[test], minlength=2)
            fold_row = {
                "seed": seed,
                "fold": fold,
                "cv": config.cv,
                "n_train": len(train),
                "n_test": len(test),
                "train_negative": int(train_classes[0]),
                "train_positive": int(train_classes[1]),
                "test_negative": int(test_classes[0]),
                "test_positive": int(test_classes[1]),
                "test_raw_min": int(raw[test].min()),
                "test_raw_max": int(raw[test].max()),
                "estimable": bool((train_classes > 0).all()),
            }
            fold_rows.append(fold_row)
            if not fold_row["estimable"]:
                return _empty_result("temporal_support_nonestimable", config, matrix)
            scaler = StandardScaler().fit(X_seed[train])
            model = LogisticRegression(
                C=config.C,
                penalty="l2",
                class_weight="balanced",
                solver="liblinear",
                random_state=seed,
                max_iter=2000,
            )
            model.fit(scaler.transform(X_seed[train]), y[train])
            scores[test] = model.decision_function(scaler.transform(X_seed[test]))
            for local_cell, coefficient in zip(cell_index, model.coef_[0]):
                coefficient_rows.append({
                    "seed": seed,
                    "fold": fold,
                    "cell_index": int(local_cell),
                    "cell_id": int(matrix.cell_ids[local_cell]),
                    "coefficient": float(coefficient),
                })
        if not np.isfinite(scores).all():
            return _empty_result("score_nonestimable", config, matrix)
        seed_rows.append({
            "seed": seed,
            "auc": float(roc_auc_score(y, scores)),
            "n_trials": len(y),
            "n_positive": int(y.sum()),
            "n_negative": int((1 - y).sum()),
            "n_cells": len(cell_index),
        })
        oof_columns[f"score_seed_{seed}"] = scores

    oof = selected_labels[["trial_id", "trial_index", "late_hit", "miss"]].copy()
    oof["y"] = y
    for name, values in oof_columns.items():
        oof[name] = values
    score_columns = [name for name in oof.columns if name.startswith("score_seed_")]
    oof["mean_score"] = oof[score_columns].mean(axis=1)

    coefficients = pd.DataFrame(coefficient_rows)
    cell_summary_rows = []
    for index, cell_id in enumerate(matrix.cell_ids):
        selected_seeds = sum(index in indices for indices in selected_by_seed.values())
        values = coefficients.loc[
            coefficients["cell_index"].eq(index), "coefficient"
        ]
        cell_summary_rows.append({
            "cell_index": index,
            "cell_id": int(cell_id),
            "selection_frequency": selected_seeds / config.n_seeds,
            "median_coefficient": float(values.median()) if len(values) else np.nan,
            "mean_abs_coefficient": float(values.abs().mean()) if len(values) else np.nan,
        })
    return DecoderResult(
        status="estimable",
        reason=None,
        config=config,
        experiment_id=matrix.experiment_id,
        feature_name=matrix.name,
        seed_metrics=pd.DataFrame(seed_rows),
        fold_metrics=pd.DataFrame(fold_rows),
        oof=oof,
        cell_summary=pd.DataFrame(cell_summary_rows),
    )


## 3. Load and verify all ten feature shards


In [ ]:
from contextlib import nullcontext
from functools import lru_cache

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import HTML, display
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")


def display_matplotlib_figure(figure):
    # Emit a native PNG payload that is reliable in Colab.
    try:
        figure.tight_layout()
    except Exception:
        pass
    display(figure)
    plt.close(figure)


cache = load_feature_cache()
index = cache.index.copy()
display(HTML(
    "<div style='padding:12px;border-left:5px solid #d97706;background:#fff7ed'>"
    "<b>DEV-only educational explorer.</b> It uses fold-independent features from "
    f"<code>{FEATURE_TAG}</code>, never downloads raw neural bundles, and never "
    "requests CONFIRM data.</div>"
))
print(
    f"Loaded {len(index):,} active experiments across "
    f"{index.ophys_container_id.nunique():,} containers and "
    f"{index.mouse_id.nunique():,} mice."
)


@lru_cache(maxsize=24)
def experiment_data(experiment_id, feature):
    experiment_id = int(experiment_id)
    matrix = cache.matrix(experiment_id, feature)
    labels = cache.labels(experiment_id).set_index("trial_id").reindex(
        matrix.trial_ids
    ).reset_index()
    q2 = cache.q2(experiment_id).set_index("trial_id").reindex(
        matrix.trial_ids
    ).reset_index()
    return matrix, labels, q2


def outcome_labels(labels):
    return np.select(
        [labels.late_hit, labels.early_hit, labels.miss, labels.aborted],
        ["late hit", "early hit", "miss", "abort"],
        default="other",
    )


## 4. Cohort browser


In [ ]:
def metric_cards(items):
    cards = "".join(
        f"<div style='flex:1;min-width:150px;padding:14px;border:1px solid #e5e7eb;"
        f"border-radius:10px;background:white'><div style='font-size:12px;color:#6b7280'>"
        f"{label}</div><div style='font-size:26px;font-weight:700'>{value}</div></div>"
        for label, value in items
    )
    display(HTML(f"<div style='display:flex;gap:10px;flex-wrap:wrap'>{cards}</div>"))

metric_cards([
    ("Active experiments", f"{len(index):,}"),
    ("DEV mice", f"{index.mouse_id.nunique():,}"),
    ("Aligned trials", f"{index.n_trials.sum():,}"),
    ("Cells per experiment", f"{index.n_cells.min()}–{index.n_cells.max()}"),
])

figure, axis = plt.subplots(figsize=(11, 6))
sns.scatterplot(
    data=index,
    x="n_trials",
    y="n_cells",
    hue="project_code",
    style="session_type",
    size="n_cells",
    sizes=(70, 360),
    alpha=0.8,
    ax=axis,
)
axis.set(
    title="Feature-cache coverage",
    xlabel="Aligned trials",
    ylabel="Cells",
)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
display_matplotlib_figure(figure)

hierarchy = index.sort_values(
    ["mouse_id", "ophys_container_id", "ophys_experiment_id"]
).copy()
hierarchy["row_label"] = (
    hierarchy.mouse_id.astype(str)
    + " / "
    + hierarchy.ophys_container_id.astype(str)
)
hierarchy["experiment_order"] = (
    hierarchy.groupby(["mouse_id", "ophys_container_id"]).cumcount() + 1
)
figure, axis = plt.subplots(figsize=(12, 7))
sns.scatterplot(
    data=hierarchy,
    x="experiment_order",
    y="row_label",
    hue="session_type",
    style="session_type",
    size="n_trials",
    sizes=(80, 360),
    alpha=0.85,
    ax=axis,
)
axis.set(
    title="Mouse → container → experiment hierarchy (size = trials)",
    xlabel="Experiment order within container",
    ylabel="Mouse / container",
)
axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
display_matplotlib_figure(figure)

coverage = pd.crosstab(index.mouse_id.astype(str), index.session_type)
figure, axis = plt.subplots(figsize=(12, 5))
sns.heatmap(
    coverage,
    annot=True,
    fmt="d",
    cmap="crest",
    linewidths=0.5,
    cbar_kws={"label": "Experiments"},
    ax=axis,
)
axis.set(
    title="Session-type coverage by mouse",
    xlabel="Session type",
    ylabel="Mouse",
)
display_matplotlib_figure(figure)


## 5. Logistic-model workflow

The registered analysis uses regularized logistic models with temporally isolated evaluation and training-only calibration. The diagram emphasizes model inputs, fitting, and metrics.


<?xml version="1.0" encoding="utf-8"?><svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" data-d2-version="0.7.1" preserveAspectRatio="xMinYMin meet" viewBox="0 0 1323 2239"><svg class="d2-3478767953 d2-svg" width="1323" height="2239" viewBox="-13 -13 1323 2239"><rect x="-13.000000" y="-13.000000" width="1323.000000" height="2239.000000" rx="0.000000" class=" fill-N7" stroke-width="0" /><style type="text/css"><![CDATA[
.d2-3478767953 .text {
	font-family: "d2-3478767953-font-regular";
}
@font-face {
	font-family: d2-3478767953-font-regular;
	src: url("data:application/font-woff;base64,d09GRgABAAAAABOEAAoAAAAAHWAAAguFAAAAAAAAAAAAAAAAAAAAAAAAAABPUy8yAAAA9AAAAGAAAABgXd/Vo2NtYXAAAAFUAAAA6gAAAUhSSkS6Z2x5ZgAAAkAAAAwwAAAQ2DA1Y3poZWFkAAAOcAAAADYAAAA2G4Ue32hoZWEAAA6oAAAAJAAAACQKhAYCaG10eAAADswAAADQAAABAHWMDUZsb2NhAAAPnAAAAIIAAACCmaqVTm1heHAAABAgAAAAIAAAACAAWAD2bmFtZQAAEEAAAAMjAAAIFAbDVU1wb3N0AAATZAAAAB0AAAAg/9EAMgADAgkBkAAFAAACigJYAAAASwKKAlgAAAFeADIBIwAAAgsFAwMEAwICBGAAAvcAAAADAAAAAAAAAABBREJPAEAAIP//Au7/BgAAA9gBESAAAZ8AAAAAAeYClAAAACAAA3iclM+5SmNxAMXh7yZ3tkwmM5NZ3PUaNSYuUeMuWlhYBCsR7C0FSxHUzmfQ1xBJ6wJ5EaMPIYgE/kJIY+mpv+J3EEmLkBVH66hIxHISZZOmTKtatGbDlpodew4cO1VPXyTF5Cg5K+RDQKLU8bMdv6lm2659h07e+/AkIyslDq/hJTyHVmiE+3AXbkM9XIercN66fMw0G82bh2K77iOLVM1bVjFnotM0Y1VKWuyTz7746puM77J+yPnpl9/y/vjrn/+6dOvRq0+/AYOGJIYVjBg1pmhcyYJy++mSFd4AAAD//wEAAP//gvI3bgAAeJxsV31sG/d5ft8fKdKyKEsn8nSiRIq8O4nfFCUe704SqaMkfkjWFyVSsvUR0x+RLcm2UkdeEzhWbBRxnKzJWi5IlqBzirTLgGao13YFnA7BgC1uMm9tkjYoUmepjWAYtGL9nCZszVqRxR0pWW7z1xHE796P532e5/0dVMEcABHJ86CDaqiDBqABBIql2lm3mzfKgizzjE52I2Wcw58UC4gHI3pJ0ncN/mzwwuXLOHuJPL99tvfK0tJb+UcfLf7Zxk+LYXz3p0BAB0DspADVQAGYjYLb5XLzBoPOLJh5N2/8nuMtR4OzXl/n/Le7+btzyq/i+JnFRXm1p2e1OE8K2+du3QIA0ME8AGkjBaCgGXi1NiHc2EhbDEZaexh4nRCWxIiL56mdH/M3Eyd7ukLR0fi5kUvHp0fGx0+uzeSPHFojBWe6tytTp6+ZSPYf8uGF3nBP5/ZWfLCvBwAQIqUt0kKugR2ginO5xIgkCeFGxuhy8ZzBQFsaG4WwJDMGA2aznxsdu5KLPWALNg/6lCNCeEEJjTg63CdMUy+dOf1Stssp2biBR7LZC4MeLhIMAwDReomQAuxTMdE6oS0G3r1b96svffnlF2ZGz58/f36UFF679vLfJp9ZX39Sq20eAO+SAtRo86FZWqB5mqXn8bHih598gl2kkH536JdDO2fhOimoMxAogZrPqYCW/ydxUgBT+X8BBaOZ1xnp+ZwOqfw7vzjy3YdJofg6Hvz/4mmcefIHO3m/TwpQVX6Hpedz6CCF7dfVVJWYj5OCiplACebGRkaQJNksUDwVkWTeqON1br6xkabmFy+ZGJPeRJvWT07s0+kj6/J6RK8zkkLxq1yK41Ic5rfP4UrgjP+F4tdx+gX/mUDxxd0cQVIAczkHI7hcIiVQu5EP/WJIrzNmDv1ySK9X4y0+HT4Twdz2OXz5qa7lSPE1INpsT5JrUPcH09VI5A5L2gg4bcg4lr08NHQ5m7s0PHwpFz3ceXp29nTnrGn6SysrL05Nvbiy8qXpg4kL2ce+8IXHshcSsDvbGm0+lj0s5XnqHi3fGHlIuXr27IlDucOH8qTQNjO8tFj8HQ4PpIdkLUaktEDOk2tQCz6A9vtqquJc7g6yt2bjPTaO7daIuoXH7ENnY+Xqe5fTzIzYyNXW14V7Ehen1YKnLybuziX2zz53rNxLvjBn8geq9P2Gag1rX2kLf0WuQVBDyS1rWcSIy+W+l7+iCLUuhmklajVYn3rEH+aPCgPD9i5H3tHnFfPR6CIfbD3YISfYcPMRV1+btGgSA73twWgn57Ed8Nb6BjvDmWCwTbKzkYDD21zjqQ8OdEVmwoBgA8DfkQIYVTR5kaV56t/fxo/fJiPp9PaNMvcOl7ZIBymonqVNlBKoskYl7afBgInEGSXnTfkDae+kctokra/g54qPZxZcroUMPlG8vLIuAWoD1JEC1AIIuj0c1r3//txKQ4tZ32CjVmZ+QArFL/ee7O092Ysnts+VOYXfwk1ohjYAhlMpJUc0aIxuDSia4lXTc4clWdQM5M2+qS/+JeX3+EbsTu7B3rnJpFHHTTXyCn/heNh0cGByhnJ0805LT6N3daH4Qa/NN8g5nqqLhbztQCBb2sLfkltgBmd5OryRpwTaWM5V5nGZMqo7opc76NQZB7OEzXiOnogeTccy0ZSjn3fGTaw9TG69OWt3X30494iSWpqffJBzlmxMGdeO0hZ+AzfVGXy6D+4Qr6F/OTZwRulMWX10yB5IuXMJrrexjZ00xdYms2sxjpHMTaGZ7tyS3SLbWZXjodIWfrjTQxkzLbhbFHbAksXdRL9ZeCh6XPYpTn0uadTZxqz9MUdPqzvuSpuevJA5r7Q2597Y7u6xeVOJoo0J5boPPwhEq/9fcROawHFfB+raYHdlo2M1qJAZOK3EF+UjJ5EUv1N1OM1HW+yOzPdQH+8Rpkx9a5nJNWV9udZaPf4ATUmWVnSNjGc0nFoBME5+VN6ZvCiLkQpOPEer/kwdGxxMHWR89Q0ttuTSEv6VUjU+crjaGDflxxPFI9p+C5ac+HPchC7og/FdFomuPQ8tqEDzlYXHucszqMxcF75nE+aK1XCu8pn/nTvnYhusnLnJHZ7usrTVvrZIMZ2TYTdX29DelZ+ZiT005uuL+f2xPik9LYSmD7D1zU2jHyfjjp5GfY3H5uio1VuSfnHCZ6yK14uOyJiXqmmxMK1yX3AshN+Ki2IsJorx4tN9Lq5Zrzf7aHeHhk0WAH9MblXccIej6lbQ+Ellszp+PDw+lA10tkfbya03F9nQ8SPF76M3qbjai69AqQQpAPg2uUFc0AsABoiul/mZLW3BbXIL6sp4abKvDPW1Dm/2QLXeaKzZ12jqEcmp7efNFKKi15drIv+Nm8BqNakiV5G9rzLj7jObNOqcY/7ueJ1rIjB6MBvokJLZQEhK4kaaD3UFvJGdckeLr1QeO33jZqXvSo69fSeNOn5it3Et2H19V/j7a9yEOmj51F21O2+siy7F40vR2Kl4/FQsPj4eVyYmKtqLrWUn12LJpdz08vJ0bgk0/xDwt7hZ0d696jRWudwMbd7rH2qlbMafPxE92s0lOPKoZh/xNlZ5h3y72+Z56uHsI0pr88yraLjPP1SNC/jhTp4qUdbC7xJZFijdXo3jVb191FcWej9L9g2+tyvyd67P2jya0O32ju1xNNxT+Q7H8rhZuU2Vu6m4VBlo67DXztSbLHWOhBU3Zjuk/cN6fVgp3irzyFbawidwU922zP27Tlt1f7Dpyovuh5E873Um/Z2drNDCDfrmMsEJm8cqOTv8rZ0tfDLozZjcNtnKBh1Wjtlfy4reaMbJRMxNPhtjp2tqWbnDPejR8jeVtjBFHgKmwmNelGVBM45dPv9som94bH/qiSdYX22rqd4SMs0PY61S9fTTieJmsKtarxhrtFijpS18FzdU3t2nCapiqx+PD+f8na4op+LCjZmOH8FI8cdJxe3HuWLzmKcTUL0X4j/jxh/vwTe+MfNADVOjr2H2PzD1ddwo/rxtmOeH29BSbFbfK4W091r24ijL94U4QObr7ab6fZZqr1RXc3PmwRprjb7Gsv/w5OtUKPVDg36AVEWDbfifxf9xDHPssBNrtzc7x4Jqbw4A/CJuQDWAIKJ6GUCWdiD8B46VAPcF8NFEoPinCY17rQDanaBKuymLZsFMt7M6N29svX3sdnEE50+l9KkwyW9/NTyEgfffV98JlLbwLfJ5qNmZRKQii71a++TY6uqxo6urR7uTye7uVMp0/ZWvfO1rX3nl+uDlZ5+9ePHZZy9rc8gA4OvkknbHVleaKEmyaoCZ5/4kMNAcv5LED8R9TP3228kyB9sA8Lvk8yp2gqiQivzcu8JUjVOgPceupmN9nqQt5FlQ5k4lPjvW3G39+65jf/5ZQU4HnaGAuDQTu/hUhuiHAMFa2sJ/IpcqG/Yer7XIZpbmjffs5L/GFlmPfaw7OjWisCF7gMb4/1FMh12ek/pOmCRWsgUzicERi9mGwtA/mA74Z1Op4+Fy7aHSIXgb1qABgHFLktvA8XvAS1j8nUgMpIlvszrb03/TaY570G5rcUSC/cdB9feB0hZ8Bz8ibogB4AoY1GepBPnSlu48rIELYnYAN8TwLAAYIUbe0PJ64SOsw2b1+0YWBdq78VE8Xt4Lk1hNfqLylykvUEZzGOYDJZ1WhN6ent5vnrxz5crdxaajd9bW7hwFBFdpEu5U3nFrN24VL9pimNPOC0o6/c3K6abFu1eu3CmVwIPPYL/OTozQB4DPaDX78Bmc1FHECMruf61ExilylhghDkASWu0cLON7JKCyUxZ5URA1wdO3b9wYuHFj+aZy86ZyUzuHfnwPV9QezSJLc3gd/YoCWlwO/hHfwx8RF3TDGTBAN7yw44fwKm7sfPdls7ih6rP0L2QEZHJDzUntGVCTw9HU5HCQEbu1qbW1yWpXY2gzgTX17F4lPG7leWsTz5v4FjvP21t49awHAPvxOVU1gihJf8xdgY4s5D281Ue5mIQnOuQa8P11e3Yp4AqwTTzn61faU8P+8rePDwAn8SpUQT2ALIgyb2REXiz7Ib36Gb0+HP67V4d7e/8ivTr1m2V2+9cdy9odrLSFU+BVe2ZElm5FwSSKvwcAAP//AQAA//8V75O6AAEAAAACC4VsgL7VXw889QADA+gAAAAA2F2goQAAAADdZi82/jr+2whvA8gAAAADAAIAAAAAAAAAAQAAA9j+7wAACJj+Ov46CG8AAQAAAAAAAAAAAAAAAAAAAEB4nEzLIUtDYRyF8eecGyyCTSaM60WHuKm75aJBDCImB8K/iK9gNd5PISa73WT3Q8xs8BuYvWVsaeKLguFwyu/xI9dMwRWFJyQf0/iO5DWSVkg+I+mT5A+SH0h+ovHh768zco++77lyAZrTeEhoyth71PpirAGl5hy4IphxzpIoTghvEy6zi2xvCD3TV9BzxYXeWc17YdMTSnXsq+NSHQN1bLCgZsFp8cqtjhiqJlSzo5ZdtYzU5mbrb8wIWL79NP/NNwAAAP//AQAA//956zAmAAAALAAsAFAAhgC2ANQA6gD2ARABIAFSAXQBpAHGAggCTAJeAoICngLWAwoDOANqA54DwAQsBE4EWgR2BKgEygT2BSoFXgV+Bb4F5AYGBiIGUgZoBoIGqAbABuoHHgc+B0oHWgdmB4AHmgekB64HuAfMB9oH5gfyCAgIHghECGAIbAAAAAEAAABAAIwADABmAAcAAQAAAAAAAAAAAAAAAAAEAAN4nJyU3U4bVxSFPwfbbVQ1FxWKyA06l22VjN0IogSuTAmKVYRTj9Mfqao0eMY/Yjwz8gxQqj5Ar/sWfYtc9Tn6EFWvq7O8DTaqFIEQsM6cvfdZZ6+1D7DJv2xQqz8E/mr+YLjGdnPP8AMeNZ8a3uC48bfh+kpMg7jxm+EmXzb6hj/iff0Pwx+zU//Z8EO26keGP+F5fdPwpxuOfww/Yof3C1yDl/xuuMYWheEHbPKT4Q0eYzVrdR7TNtzgM7YNN9kGBkypSJmSMcYxYsqYc+YklIQkzJkyIiHG0aVDSqWvGZGQY/y/XyNCKuZEqjihwpESkhJRMrGKvyor561OHGk1t70OFRMiTpVxRkSGI2dMTkbCmepUVBTs0aJFyVB8CypKAkqmpATkzBnToscRxwyYMKXEcaRKnllIzoiKSyKd7yzCd2ZIQkZprM7JiMXTiV+i7C7HOHoUil2tfLxW4SmO75TtueWK/YpAv26F2fq5SzYRF+pnqq6k2rmUghPt+nM7fCtcsYe7V3/WmXy4R7H+V6p8yrn0j6VUJiYZzm3RIZSDQvcEx4HWXUJ15Hu6DHhDj3cMtO7Qp0+HEwZ0ea3cHn0cX9PjhENldIUXe0dyzAk/4viGrmJ87cT6s1As4RcKc3cpjnPdY0ahnnvmge6a6IZ3V9jPUL7mjlI5Q82Rj3TSL9OcRYzNFYUYztTLpTdK619sjpjpLl7bm30/DRc2e8spviLXDHu3Ljh55RaMPqRqcMszl/oJiIjJOVXEkJwZLSquxPstEeekOA7VvTeakorOdY4/50ouSZiJQZdMdeYU+huZb0LjPlzzvbO3JFa+Z3p2fav7nOLUqxuN3ql7y73QupysKNAyVfMVNw3FNTPvJ5qpVf6hcku9bjnP6JNI9VQ3uP0OPCegzQ677DPROUPtXNgb0dY70eYV++rBGYmiRnJ1YhV2CXjBLru84sVazQ6HHNBj/w4cF1k9Dnh9a2ddp2UVZ3X+FJu2+DqeXa9e3luvz+/gyy80UTcvY1/a+G5fWLUb/58QMfNc3NbqndwTgv8AAAD//wEAAP//B1tMMAB4nGJgZgCD/+cYjBiwAAAAAAD//wEAAP//LwECAwAAAA==");
}
.d2-3478767953 .text-bold {
	font-family: "d2-3478767953-font-bold";
}
@font-face {
	font-family: d2-3478767953-font-bold;
	src: url("data:application/font-woff;base64,d09GRgABAAAAABOAAAoAAAAAHTgAAguFAAAAAAAAAAAAAAAAAAAAAAAAAABPUy8yAAAA9AAAAGAAAABgXxHXrmNtYXAAAAFUAAAA6gAAAUhSSkS6Z2x5ZgAAAkAAAAweAAAQmGCWMc1oZWFkAAAOYAAAADYAAAA2G38e1GhoZWEAAA6YAAAAJAAAACQKfwX/aG10eAAADrwAAADXAAABAHxrCyBsb2NhAAAPlAAAAIIAAACClwSSvm1heHAAABAYAAAAIAAAACAAWAD3bmFtZQAAEDgAAAMoAAAIKgjwVkFwb3N0AAATYAAAAB0AAAAg/9EAMgADAioCvAAFAAACigJYAAAASwKKAlgAAAFeADIBKQAAAgsHAwMEAwICBGAAAvcAAAADAAAAAAAAAABBREJPACAAIP//Au7/BgAAA9gBESAAAZ8AAAAAAfAClAAAACAAA3iclM+5SmNxAMXh7yZ3tkwmM5NZ3PUaNSYuUeMuWlhYBCsR7C0FSxHUzmfQ1xBJ6wJ5EaMPIYgE/kJIY+mpv+J3EEmLkBVH66hIxHISZZOmTKtatGbDlpodew4cO1VPXyTF5Cg5K+RDQKLU8bMdv6lm2659h07e+/AkIyslDq/hJTyHVmiE+3AXbkM9XIercN66fMw0G82bh2K77iOLVM1bVjFnotM0Y1VKWuyTz7746puM77J+yPnpl9/y/vjrn/+6dOvRq0+/AYOGJIYVjBg1pmhcyYJy++mSFd4AAAD//wEAAP//gvI3bgAAeJyEV2twG9d1PvcCxIog+MBjsQCI5y6xC4AiSGCxWJAECT4AkpIA8CWRlMWHzFEkSqQkVqJCOqbizFiWpw4Up6asyHXaJBq7D9dq5VE7o2SqdMZpkmqizrh1Us10GlltPek4nbFZD5vWLgl07i5Iif6TH9jLWd495zvnfN8590IFDAHgWXwVNFAJtWACGkA0+ox+URA4ShZlmWM0soCM1BA2Fd98Qwhqg0FtyHvd85WZGZSbxle3Fo7kZmd/M9PeXvzD7/+geAWd/wEALn0OgHtxASrBCGCmRIHnBU6n05hFMydw1K/qvl5bXV+tNdg/v//O/W8HfhpA+5PJyBkxdrr4Ai5sLb3+OgCABnIAOIkLYAQHsASbGLVaaYuOopVFx2nEaFyK8RxnFKPKmvsgvdDVFIj2ps/1z2TikWisb/SZZMcoLrj6Uo2jtdrqA929B4PoxRDHe4sTE41+AATh0gZuwdehHqCC5XkpFo+LUStD8TzH6nS0xSpG4zKjQ1MjL40eujKSOubL22Vu777GsYFAypYfMWRfPb3w2rDITjOu6HTPscUG++RRwAr+LC6AXs1sGb2OE8RonOAmgO8ce2V46OWjTc7EaDg8mnDiQvrlxcVX+pcDk/n8YT8QfDkA9AkuQJVSH9pHizRH++gcul78v4cPUS0urD7/7LXV7b3wCBdAo3g05tZIQtX3eAEXwKC+F82ixsxpKDq3pn33xo8/+t53srhQ/G9UVdwsriDzsb/Y9vtvuAAV6jc+OreGMC5srRNXZZs3cQE8yv/NVisjxuOyWTRyJIUyR1GcIHBuTNO5753Um/RavVF/4ruXqUqNVpoanopptXsoXCg+dHa63Z1OxG4tfeIdHPK8/tlnr3uGBr2fbPsgOTSrPhiR5yVJNHIagbNaaTr3rbe6tNqaAlkqqnGh+NffjH2t7VdbSyjzjfhq238AAFbq+1V8HWq/UGGlGoJKIZbUGY1NvHDgwAsT6rM3n+/tzecNI6+dmn91cPDaqVOvjTy3NDt75szs7BKU69ui5NSyq74cvcPHRwMX+vqWMsMDK13JNC4Ik4PZ2eZfopE5MQRlbPvxd/F1qAIewP8kFpYXdiGlrGUe6tBYGRpynPgmN3dRRbw4wwwGaX+Nw9S+cGGOoJy7UHz4paz1z19S4T//Z/Wsm9LOV1YDAq60gfX4OoSUjAiyYlyK8YIQxrsFQFusDKNGhixdz0UPcmOBcJPYeMiX5NtPphOLoQPeLoFvag0dbO9rO2NoCX/JzbMuj8vUUNPc1xyfiO0NTdnrPU6328jaDmbikwlAYAfAZlwAimSOk3w0Z7x/G31+G9etrm6tq/zaV9rAg4r+CUbJKBoVKSp/6FD+uctX22Q5+Y3nDdfeQNPFtaPZ7FF0unjjjWuASp8BYBEXoBpA1DzBTc0Pf/ztfC1Tq62x1eSu/S0uFN+TjsfjxyXUsrUEGEKlDfRztAl24AAYlhRAVtJACUpSaCNHepkcjcuS0ht+mB66tIa5oKerQWqeb5s5vqLXevr32P3mfNJjGE/lJ2p9go1+2tVw5lzxQ9HJnWPM4/pGl41R6t9d2sBWfBcsREWkEhzFGUWa+gJBOZY0PZTx9bq0hvNrWleaTU40J2cm+PjY3qAlYPB5JXz37azD1fk72UPPpFb6spebfmaqUfLYUNpAd9EmOL7Y39TqbrPKnjnbPfDldLjfmeG8UirVYgub2/xjho4LI6NLHW5mxpXt7srRtUe99Sp3hdIG2sR3wQze7VwphgUi0J0sbZPo08mz7TOxYMKuW1vRax192CaYzI0WLt5s+Pozwxc6nbbsn271RhzcisX+M1NNb/++DGAF+7+iTbCV87NLET7CWoJdIyqyQZ7+cz29C+39U81aXHyg74tI8Qg//fu3hb1s3NC5NDK8lErNp83+yrjoO+xwo7ag1KxyzQaAlvA9shI+yl/QAGm5xqd6ehqGej2xuvpqh6Heffgwuni6ol4aixl0CxUVPt59vvg8mVtsqQlTaBOaoR32K5nhpRhJBCGTtB0CI9JcuWGwglIHQi+LTqd5oguYywOO5ZUtn7ZNJ/rN9V6bI9g2Le31/dUgVRmbkF0eExscmnw6vbrfJQgulyAEo12CX7T7DPUd7zsSe5MBbXXAUx+t05rSjcnBgGG+irW07m/Q11rNpvZecTiM7oWCQjAQCIaKaw12pk6jsdmdLjU33aTYCkdJlytzkzZyRgUlZexeo5wHosP71lxeZ8CG77592N44P1W8j3zxgJ0pvgOlEsgA8Ev8PuahDQAoaIeXVNulDWTCd6FWZdC2xklR/y7bvmasrKB0JoPfcOQA5rYeMCaETldQKiaNC22CT8FExE2qtQsZtbN2E032RaRus29/ZOjAmsvrbyGPZrTe5WlqDLCRbbgtxXfKy3bcaLMcd9nHk3Gv6LXe3E7gaD3lbtoVt8pfhQu/ffZYU2fT6bOp1Jl0+kyqKRxuCjc1lbXXsTQ6cqFjOdfVnSUSVPvGALaiTTCDG4B5jE6hEy8wtPlx2yA4XfuEp+aSM3Fv0lExyMfHGkOWwB38JxEH97vnD62k6u2Dv4cadpoG0fYA2lTsewEqJFkxuy0KURaNmie1jU7q7D2sKvBO0qE+3BH3nW9lbR5F4C5vZGsCNTxWd5lb6GW0CaZddVRVp2a4PsvTTr2t2l7n7LCg9fFopKLiOa02GC0+AgR0aQN9B22CoPDn8Rzj1Tm2Y4xMMTemLbr3Iyf4Hjbl8bldYYe7PXDyUOu4p8cRc7S28t6O4JyB90za6xmz0WrWGxpag5kxwTZhsQo2e00V1xrunVI1YSxtoDN4CRh1NkmcJMuicjB73FBhcjCdNX5leZlzGex6xiwbTo3dO627dOn8T0N+nXZeZ1BtJUsb6H/ROuHZLg0Yy230n4b3rbm9Tt66tlKl8ew3zE+hWPEDKehwoYFiXca/FxA516ESWi/PO6Y872RRc/uPr3bpzXptpVnffeUGWv+1PycIOf+vi3WKb0OpE22hdcLOx/mT5V0mavCK1VfroEx7/AE99TdX+6tMeu0eY2XyyttMYvBdnXYRVTS4HOjff8H2+bl+7hfFqs5DITU2HgD9JVqHSgBRMpNBrxFp/r3vo8X3Hgyi8Pl88R/OE76xAHhaOWtWKTtFM63eQ9if3PpJcXYzk9DKh7G49feHV//lRz9S54+/tIH+E78IVWVtqfWmLURX6vlaPb5b0Z7jFy8eJz97gGECdlvAZgsY3rpx4803b9x465x/enx8kmUnx8enlftAHwD6Z/ysclYmo0yKx2XS+PpeWo4NsAvLy+jsEb3TsrW5rMboBkAf4hfBSfZ3YlXS5fOCokjSMUXaP3yxLxJkZdtQ82w6NS21T8ZsSevXDuYunmxqjgiOwagYPdIhnT0b11SsErtMaQM9ws+SybeL24p1s4/mqB0P/5Nf4Htc6UCkLbHX6Xf1mNDcR1U+Xj6S6D5liPmnHP5opCVaYwqh7tXl2tB4uu9YDNS7TycCuEn0xwjxuMCy3BOJzLoTbQhrMReP89HY5Lt5S7e/McCH93ePrACQvt5R2oBP0H9hAZIAKA06spZKkC1taF6Fm8BDshZAgCQ6ovT9JP4jxW8j3EM+FCH3FFkS6cbf3JubI+8HSjkUwB8QHjPqwGSUGjL3U5lMalKORuXbJx5euvTwBP/0g/lTD2YBQUsph+rK3whK5UmuaIuuMJmIRhOTqUzmNj/74NT8g6d55dtSCerRMsprKjEFHQBoWcHMoWW0iLcwBZ077xyYRjP4JKYgBYAlBTsL0+hjHCdMlSVOEiVV+P9469bCrVvTd+bu3Jm7o+xDVvQx+iqJ0Sz5aBbdR1YSJbHLwg30MfoU85CAJdBBAgrb/RB+jta372/da2i9WAeodBO3wih+n/g0PlEgfzjs94fDuDXEcSHyIzaUmsBNspd5Yu81XhR5XhQNkhCQpIAgkb31ACiPXlFuqFJ8h13lTyiRbjpw1F1v5owdycPdg6ngl9s6Z/wur92aGMqkmsfDTwEmJ2W0iC5DBdQByKIkcxRDGqKalstDOm2Av33lclvbHwQv5D4ej3/2Ud848e0oPUIzECOxMpKPdqCM9+DB/wcAAP//AQAA//8VrYpyAAAAAQAAAAILhfylJ4lfDzz1AAED6AAAAADYXaCEAAAAAN1mLzb+N/7ECG0D8QABAAMAAgAAAAAAAAABAAAD2P7vAAAImP43/jcIbQABAAAAAAAAAAAAAAAAAAAAQHicTMs/SgNBAIXx771AQAw4QvwPKZKoYMbFTsFsMU1QyICFQixsPcOChZ7APnfQxtYLWFjpaUyzksXC4vGa3+c3rvgAl/XCd2RPKPxA9pCsY7LvyW6TvSB7TvYLha//PjJwZMdzLh3rH69x5JKkb4YuOXSboW7Y9iZ9n5PU5VQ9UuuW5DHJo8alpdUzSe9s6JF1nzH2Kh2v0NEnB36i78C+AxMHeg5sKVIoUra+mGrKSDMuNONEFXuqGKhi16HpmqlLgvp12fw3vwAAAP//AQAA//8BaSXuAAAAACwALABQAIQAsADUAOoA9gEQASABUgF0AaABwgH+Aj4CUAJuAooCwgL0AyADUgOGA6wEFAQ2BEIEXgSQBLIE3gUOBUIFYgWeBcQF5gYCBjIGRgZgBowGpAbQBwIHIgcuBz4HSgdkB34HiAeSB5wHsAe+B8oH1gfsCAIIJAhACEwAAAABAAAAQACQAAwAYwAHAAEAAAAAAAAAAAAAAAAABAADeJyclM9uG1UUxn9ObNMKwQJFVbqJ7oJFkejYVEnVNiuH1IpFFAePC0JCSBPP+I8ynhl5Jg7hCVjzFrxFVzwEz4FYo/l87NgF0SaKknx37vnznXO+c4Ed/mabSvUh8Ec9MVxhr35ueIsH9RPD27TrW4arPKn9abhGWJsbrvN5rWf4I95WfzP8gP3qT4YfslttG/6YZ9Udw59sO/4y/Cn7vF3gCrzgV8MVdskMb7HDj4a3eYTFrFR5RNNwjc/YM1xnD+gzoSBmQsIIx5AJI66YEZHjEzFjwpCIEEeHFjGFviYEQo7Rf34N8CmYESjimAJHjE9MQM7YIv4ir5RzZRzqNLO7FgVjAi7kcUlAgiNlREpCxKXiFBRkvKJBg5yB+GYU5HjkTIjxSJkxokGXNqf0GTMhx9FWpJKZT8qQgmsC5XdmUXZmQERCbqyuSAjF04lfJO8Opzi6ZLJdj3y6EeFLHN/Ju+SWyvYrPP26NWabeZdsAubqZ6yuxLq51gTHui3ztvhWuOAV7l792WTy/h6F+l8o8gVXmn+oSSVikuDcLi18Kch3j3Ec6dzBV0e+p0OfE7q8oa9zix49WpzRp8Nr+Xbp4fiaLmccy6MjvLhrSzFn/IDjGzqyKWNH1p/FxCJ+JjN15+I4Ux1TMvW8ZO6p1kgV3n3C5Q6lG+rI5TPQHpWWTvNLtGcBI1NFJoZT9XKpjdz6F5oipqqlnO3tfbkNc9u95RbfkGqHS7UuOJWTWzB631S9dzRzrR+PgJCUC1kMSJnSoOBGvM8JuCLGcazunWhLClornzLPjVQSMRWDDonizMj0NzDd+MZ9sKF7Z29JKP+S6eWqqvtkcerV7YzeqHvLO9+6HK1NoGFTTdfUNBDXxLQfaafW+fvyzfW6pTzliJSY8F8vwDM8muxzwCFjZRjoZm6vQ1MvRJOXHKr6SyJZDaXnyCIc4PGcAw54yfN3+rhk4oyLW3FZz93imCO6HH5QFQv7Lke8Xn37/6y/i2lTtTierk4v7j3FJ3dQ6xfas9v3sqeJlZOYW7TbrTgjYFpycbvrNbnHeP8AAAD//wEAAP//9LdPUXicYmBmAIP/5xiMGLAAAAAAAP//AQAA//8vAQIDAAAA");
}
.d2-3478767953 .text-italic {
	font-family: "d2-3478767953-font-italic";
}
@font-face {
	font-family: d2-3478767953-font-italic;
	src: url("data:application/font-woff;base64,d09GRgABAAAAABPgAAoAAAAAHlwAARhRAAAAAAAAAAAAAAAAAAAAAAAAAABPUy8yAAAA9AAAAGAAAABgW1SVeGNtYXAAAAFUAAAA7AAAAUhSTES7Z2x5ZgAAAkAAAAx2AAARtNahVsloZWFkAAAOuAAAADYAAAA2G7Ur2mhoZWEAAA7wAAAAJAAAACQLeAjkaG10eAAADxQAAADgAAABAHHGB9Nsb2NhAAAP9AAAAIIAAACCoISb8G1heHAAABB4AAAAIAAAACAAWAD2bmFtZQAAEJgAAAMmAAAIMgntVzNwb3N0AAATwAAAACAAAAAg/8YAMgADAeEBkAAFAAACigJY//EASwKKAlgARAFeADIBIwAAAgsFAwMEAwkCBCAAAHcAAAADAAAAAAAAAABBREJPAAEAIP//Au7/BgAAA9gBESAAAZMAAAAAAeYClAAAACAAA3iclM7LLqNxAMbh55t2Dp12OjOdo/OntFqHotQpLCwsGiuR2FtKLEWClXtwC7YiXQrCjSj3IBEh+YumG0vv+pe8DyIJETKS0SIqYklZsbIRo8ZU1SxYsqJuzYYtu/Y1EkdxMd6JD/K5EBArtfuJdr+sbtW6Tdv23vbhTkr69Ts8hodwH57CdbgMF+E8NMJpOAmHz8e3qeZV8+ym0NK9Z5FJVTMtSdlwS1Ux54OEpI8++eyLlK/SMr7J+u6Hn3J++e2Pv/75r0OnLt169OoT65c3YFBB0ZCSKfPGTauZ5QUAAP//AQAA//+AYTdxeJx8eH1sG/d5//N8j+LphZLMd5MSRZFHHiXySFE8kidKJCWKehcp68WSFVlvluM3WUkUx3Li2k5bG/XPSRqXNZz+sCKtu6UrWgRoAmf7o1uWYnWBuu5crEDadUibocumtPaGNIKWtV11HL5HSqI8oP+cDtLd8/J5Pp/P8z1BGbgAyJPkJjBQAbWgAyOAqHcwjChJnJkRPR6OZSWPXs+6LuPdy19WpWf+velrvxfsqv7Pfmv4PxZfJze3VvAzc5/+tHz42rFjhx4+lL34s4cAACT/IwD8KclBBWgB9Kzo4XkPp1YjinrOw7EftN+pVFWqVFZR/gd8fCYzpvv1KTy/uhpeboudkMdIbmv1/n0ABjgA0khyoAUrvRf1YshkNKjVLGtSfnKMGIpGwjy3e8NdeWNhxZd2odjXf3GkfX5+pnfo8Okz809mB8+S3FC/0COUqzSptsE5Ac/1S/7Q1oPeTChB60aI5TeJn7wKdoAyJ89Hwkkihkxmluc5Zw0xGkwmMRSVzGo1OodPRoMzlzJtY/uj+ijfvtDtcg51NKUbOdecJv3cSPbms/2St7nRk3j8uXjHXKSxLmT3U2yUnqIKNvqSjjiPGIpud/CpF16evPXU1NTkxfSJo1GS+3/nn/2rY10Hv3Rk7lShThpjH8lBlTIz1sGKLMc6WO4KLlfLH3g/rvlIRL6G5FI/7f6ku/A8bJIcMEpGhrsycoWCvBPrEMmBpvA3EUVWzzEsy10ZSTE4OP3JK2PPv+AnOflt7PmjvIJLV3+x/R7eIDkoK7xHs4+cQ0M1yW3dLuYkb5McWJS/682iRDPro1GJYxmOoXxgGe7KXMyk6rszd2U4U2HVqA78vZAwqdQ15UMkJ3/12jVc2lrFM8Ky74b8dZy9IZwS5OvF2MdJroig3ixGo0r0nagjX/Kq1DWVvcNXsjd9KnVtZR/JybMvtD4h4uzWKr72srgckm8p84jnN8k8eRX2QaMy9eLQTUZDDfGEkoTOpDB8tD+5Fpha6xs6Fg5MnU1HDiWdQyP0Oqj5/xeHc2u9PRcmhr+41puOL63Fjqx1LK21L57bmblfmZehdOYco9+l7Xdmzwx99uCpcGrh2HJm4BjJDU2NnmiVf4f9owdiIhRrnVdqrQUBwL2nOLOT9/B7ime32YpcSan/PXHeMnGo2IZv5Ilu7WS4tq66zBHpOHquvVD10rl/yozqv3C62NPA588Mqn0+FZOoAgRNfhNl8ip4AWhOSckSCfMeJX00uiMYtZqWZS4o9cP0alPMNinFx/zujLcjMtvRsWgXLX0Bd8TW6sq0hDuOa9rbfb5QT5srZApYB6XQeCjcFGhotgfr+BaTv75faj8cBoQ5ABIhOWApkpzkYDnmG2vvVOOPqr+7RrLp9NZbBY0cyG8qHDEVp6owhJZEQaG3amx4/LRaNTQyXNHV2zZjHMuM11/WnDpubLHgqvyC39mXnT2NN+TT18/TeOMAJEVyUA0gMqLeZCqyDl/uOFBXVs6oLBHrXx+Uv0Vy8s3IE9HIU2FcUSQGBDz5TfwdboCBMsG8wzGzKIkMJ3FqtScUlaQdl3mrKyMMzYuehFalTx7pLFdx0zr+gEswhupd6Yi9VXN4su/8rNjkSMjWAXdLV6Dln3mnd3Au1Jko5LPnN/G35C4YqcvTKXEspxdZVlTGs4fZipc+8CS0jKHzetZjIq6DfiV9xJWONASbnWNcwCBqmhwJcvedRZtvZoqm7vIOzonJhNf9Ie8EBHd+E2/jBtTv6W6XBUXX/PmBx4XskYgQN/n1vC04FY21N0ZNTmtWc3yu55nJFqclaDb2rKa7+6zakMG9gx3xlPSyi92fBq9dx+zjs7kieiPuR9HzNC68s9X2KHxE6eW7uAFWcJfmoyxmHeqdDcCIil3TDv9t6pR/eDYopRo0ZfL3KxrTXlvM3GAb+7M8YXTNXGRes3ykd3VcCIyG6sWazlG3RSsa7eiu2l9d32qfBAQfAL5M3gWzwulOUqoilpo745vsrErtqx1JWL26uso6raO5XLukOTqJ34yVjQ1NVFdJbGXIN5GUpylmmHfhBm6AHQKlKpUktZrbyz61mtmD3uutU5yrvrcpOVRj4Q+2JEZ9g7OtfFLL6DuP65+JcWNOn6m1nkuJDS2/4G0RszPTdZIXpibTZx8LUT4yC8fR4fP+I+9s7psOdnQUtGgHwJ+Tu8VdsMtDVlkIkTBtk7Ffzwb3qZrHhWSkPJmJq1QD9QOBXnL3YYJrSbXZXfI9FAz7q4e9Afmb+TyNCX8gtwkP7QCgho6BQi4hvwl/IHdBRzuPhAtSNxqKY3sipb6QvYSoZdQsVpo0nVoLOb31RbaC0SHpUKl26iUPcIN6HK23UK65WLR6T9WlDRzpZFX8BN/eWtYy7U5EVapkNqFS9RsHhF7aT59pwNeL64OuVqlJEFNt2gZDaU+7d7uY4QbsL63hUchoxubxwB7ElAyPArbrQ+/hBtSCrZTbBUMoHD8Kgn33wLwwNB86sCAMz3v9Y2I0RC+ak4d7n5kMFK5d3as93f3p1Z7uPuXM90lexN/iRkGnbEnFNYRTHIjV7/Gcyhc71Yx7MqDINcTH9URn/8tSz7lP3uqy+4titZ+8hVg0Hf7Xbsd2P6Liq0rOMomawSP83studDgaiHs6UOqvL94qNYf7t57lW3bsdSuLuNdcC3O5iBuwr2QuZpbfnkeVypbxW4x1+6yujD2B63NCoqKnvLNDvg+Y/2N+Ey/hBnge3Z2Prk66OQuL87XWOUvQ3MV7E81tgZgwKASG6gN60cG3RhuT4eC4JtzE25sCnNVjtyabfSm3q6HJYPXbG3idMy74e9y05nh+E6fJyo4/RyXqMqLiLCX+/J2usApj/VUZV6ruguZSjKl31lirtPtaNJ3+Wms16mJlV68m5Qc6XUNDZZnE1tLYbflN/AjXqbbNu3u2qDh90aJf31HDgK1f6M3QpdZ0UNMtae16jMrv6i2UpjgtW4c4saDBDgD8Fa7/3317uT/jUqlVKq1L/4WsvIXr8ofcMOcadKFFtirv5r+Xb8EPcB2sAKyCs2J+e6LUEHVlY41Fp3OnLLqJDE83uNat+3xG/ldLx8BPWDZWkQhx+KH8kSPLcRknarc+bskKhfifAOC3cR0qADgJ6SEERbayHNO/rMZEufx3skbAi0m//Llkgafe/Cb+mLwIWnAoKP2JQ+bdUL/bO7gYCfW5mgcXWj3psE0IKFdN29HkY39+sb/9aHLmaxf6Ej1PX+tJH+p9+lpP9yFA2it+hjyvfBNI9GQUlURGZK3VLy0+XTkpdZy9rOnC90Ma59b3urZ7+D55kb7HSUmmKEzPjmhZB1tZvnh9vkWMNKacHuFQcHzaO35xAg2awNiFpccCQtxhD/LNj/VE5hdXB7rpzID6C3menqR3+R3dDq13sBy7baLqt1PztpAp1eYdEDrDdqHRMYq+6t+EtV7LwEL6SU2nv9kR9mbFZHyf1or+7rfLNZMTmacUDeZ/lZ+FHKxQDdKKJUX0xcCGqpYYS0wNnM1aP/P1gC7usposHlfDID2L5fNgzm/CXfwl8UAcuvBpUENc+f1EfpOZghXgIR4H8EAcPwcALMTJj4vfO+9jJVroN5QkiSynea/6/e39lsqP4iHyHtQCmAtSlsxq5XvR/Kn9DunkkH95pcJQ82bXa+NrP/zbOctV+V++Gji+yNNe3s2PwoPiu56ojp4pKHAUIvQvn67Q1YZoiDetV9HxlZbjC7y+6y/G1+79Da15BF/C84yNsJCALL6k9DGFL+ElRktYSO78bpK04TVymrDQCWdIWqk5CCfwHvFSvkgRLiJGRKNo5Izvf/uN+BtvnvhB7M6d2A+U5zCA9/AE7VsfcRiD+GUMxGIFLINwB+/hTwgPEiyDGiR4RdkF/wmAv1G+DWsVhYh6lmEcTOG/APoKzMxkKx5+A/udqgqVyhr2k6NbX/HjefmHKCn/Ayj6LNzH9e1vVvuR7BKuKwJH6CfDcJvcprXrS0b/nL6BMxtsHBk2myyO/SZLI6Ay72uwQp81lzzbZ7J46k373Zp6k1WwmSwCjYsAKOAN5Rwm7Swrz87iF1lkx48FGxuNzXquPu6OdC4dea2Sb5/sd9kcdQZXnautNfnUoEg1XwGAQbwKZZSlkihJHGuUOEod+rE+crbWqkob32R/lgrGKtqMr2RWDv5+2fE/H/uWAfMf5P8LTdBEe2cjDqMNxSpR/F8AAAD//wEAAP//kOez+QAAAAEAAAABGFEWDSuJXw889QABA+gAAAAA2F2gzAAAAADdZi83/r3+3QgdA8kAAgADAAIAAAAAAAAAAQAAA9j+7wAACED+vf28CB0D6ADC/9EAAAAAAAAAAAAAAEB4nDTNMUoDUQCE4X9mK1FBsFhN84rnZsEI2giKaSwknYpiOotNJdjaeA1Lr+EBjFaCYOUBNlgHJEUUCXmSiMUw03yMb9jgBTRNr+4Q3WTfp0R9EZkQvU3UG9FPRF8TfUvbzb/WD0uacukrTvTJmdcpvUbQA4VzSg0o1KDlVeRFAkMC32xlgeBlgjNK52k8c+oSdJcmOqLtFfbU50D99Kz7NFbNpmoaqucbRmnAiDx75FwtoiKHKtK7Ko7Vo6uKC9Xs/Me7aTj/hs7MqULqsaAqffwCAAD//wEAAP//4aQ81wAAAC4ALgBSAIoAvADeAPYBBAEgATABXgGEAbYB2gIcAlwCcAKYArYC7gMmA1QDjAPGA+4ENgRgBGwEjgTQBPoFKAViBZwFugX2BiQGUAZuBp4GtgboBwAHKgdeB3wHiAeYB6YHxAfiB+wH9ggACBQIIgguCFAIXgh0CIoIsAjOCNoAAAABAAAAQACMAAwAZgAHAAEAAAAAAAAAAAAAAAAABAADeJyclNtOG1cUhj8H2216uqhQRG7QvkylZEyjECXhypSgjIpw6nF6kKpKgz0+iPHMyDOYkifodd+ib5GrPkafoup1tX8vgx1FQSAE/Hv2OvxrrX9tYJP/2KBWvwv83ZwbrrHd/NnwHb5oHhneYL/5meE6Dxv/GG4waLw13ORBo2v4E97V/zT8KU/qvxm+y1b90PDnPK5vGv5yw/Gv4a94wrsFrsEz/jBcY4vC8B02+dXwBvewmLU699gx3OBrtg032QZ6TKhImZAxwjFkwogzZiSURCTMmDAkYYAjpE1Kpa8ZsZBj9MGvMREVM2JFHFPhSIlIiSkZW8S38sp5rYxDnWZ216ZiTMyJPE6JyXDkjMjJSDhVnIqKghe0aFHSF9+CipKAkgkpATkzRrTocMgRPcZMKHEcKpJnFpEzpOKcWPmdWfjO9EnIKI3VGRkD8XTil8g75AhHh0K2q5GP1iI8xPGjvD23XLbfEujXrTBbz7tkEzNXP1N1JdXNuSY41q3P2+YH4YoXuFv1Z53J9T0a6H+lyCecaf4DTSoTkwzntmgTSUGRu49jX+eQSB35iZAer+jwhp7Obbp0aXNMj5CX8u3QxfEdHY45kEcovLg7lGKO+QXH94Sy8bET689iYgm/U5i6S3GcqY4phXrumQeqNVGFN5+w36F8TR2lfPraI2/pNL9MexYzMlUUYjhVL5faKK1/A1PEVLX42V7d+22Y2+4tt/iCXDvs1brg5Ce3YHTdVIP3NHOun4CYATknsuiTM6VFxYV4vybmjBTHgbr3SltS0b708XkupJKEqRiEZIozo9Df2HQTGff+mu6dvSUD+Xump5dV3SaLU6+uZvRG3VveRdblZGUCLZtqvqKmvrhmpv1EO7XKP5Jvqdct5xGh4i52+0OvwA7P2WWPsbL0dTO/vPOvhLfYUwdOSWQ1lKZ9DY8J2CXgKbvs8pyn7/VyycYZH7fGZzV/mwP26bB3bTUL2w77vFyL9vHMf4ntjupxPLo8Pbv1NB/cQLXfaN+u3s2uJuenMbdoV9txTMzUc3FbqzW5+wT/AwAA//8BAAD//3KhUUAAAAADAAD/9QAA/84AMgAAAAAAAAAAAAAAAAAAAAAAAAAA");
}]]></style><style type="text/css"><![CDATA[.shape {
  shape-rendering: geometricPrecision;
  stroke-linejoin: round;
}
.connection {
  stroke-linecap: round;
  stroke-linejoin: round;
}
.blend {
  mix-blend-mode: multiply;
  opacity: 0.5;
}

		.d2-3478767953 .fill-N1{fill:#0A0F25;}
		.d2-3478767953 .fill-N2{fill:#676C7E;}
		.d2-3478767953 .fill-N3{fill:#9499AB;}
		.d2-3478767953 .fill-N4{fill:#CFD2DD;}
		.d2-3478767953 .fill-N5{fill:#DEE1EB;}
		.d2-3478767953 .fill-N6{fill:#EEF1F8;}
		.d2-3478767953 .fill-N7{fill:#FFFFFF;}
		.d2-3478767953 .fill-B1{fill:#0D32B2;}
		.d2-3478767953 .fill-B2{fill:#0D32B2;}
		.d2-3478767953 .fill-B3{fill:#E3E9FD;}
		.d2-3478767953 .fill-B4{fill:#E3E9FD;}
		.d2-3478767953 .fill-B5{fill:#EDF0FD;}
		.d2-3478767953 .fill-B6{fill:#F7F8FE;}
		.d2-3478767953 .fill-AA2{fill:#4A6FF3;}
		.d2-3478767953 .fill-AA4{fill:#EDF0FD;}
		.d2-3478767953 .fill-AA5{fill:#F7F8FE;}
		.d2-3478767953 .fill-AB4{fill:#EDF0FD;}
		.d2-3478767953 .fill-AB5{fill:#F7F8FE;}
		.d2-3478767953 .stroke-N1{stroke:#0A0F25;}
		.d2-3478767953 .stroke-N2{stroke:#676C7E;}
		.d2-3478767953 .stroke-N3{stroke:#9499AB;}
		.d2-3478767953 .stroke-N4{stroke:#CFD2DD;}
		.d2-3478767953 .stroke-N5{stroke:#DEE1EB;}
		.d2-3478767953 .stroke-N6{stroke:#EEF1F8;}
		.d2-3478767953 .stroke-N7{stroke:#FFFFFF;}
		.d2-3478767953 .stroke-B1{stroke:#0D32B2;}
		.d2-3478767953 .stroke-B2{stroke:#0D32B2;}
		.d2-3478767953 .stroke-B3{stroke:#E3E9FD;}
		.d2-3478767953 .stroke-B4{stroke:#E3E9FD;}
		.d2-3478767953 .stroke-B5{stroke:#EDF0FD;}
		.d2-3478767953 .stroke-B6{stroke:#F7F8FE;}
		.d2-3478767953 .stroke-AA2{stroke:#4A6FF3;}
		.d2-3478767953 .stroke-AA4{stroke:#EDF0FD;}
		.d2-3478767953 .stroke-AA5{stroke:#F7F8FE;}
		.d2-3478767953 .stroke-AB4{stroke:#EDF0FD;}
		.d2-3478767953 .stroke-AB5{stroke:#F7F8FE;}
		.d2-3478767953 .background-color-N1{background-color:#0A0F25;}
		.d2-3478767953 .background-color-N2{background-color:#676C7E;}
		.d2-3478767953 .background-color-N3{background-color:#9499AB;}
		.d2-3478767953 .background-color-N4{background-color:#CFD2DD;}
		.d2-3478767953 .background-color-N5{background-color:#DEE1EB;}
		.d2-3478767953 .background-color-N6{background-color:#EEF1F8;}
		.d2-3478767953 .background-color-N7{background-color:#FFFFFF;}
		.d2-3478767953 .background-color-B1{background-color:#0D32B2;}
		.d2-3478767953 .background-color-B2{background-color:#0D32B2;}
		.d2-3478767953 .background-color-B3{background-color:#E3E9FD;}
		.d2-3478767953 .background-color-B4{background-color:#E3E9FD;}
		.d2-3478767953 .background-color-B5{background-color:#EDF0FD;}
		.d2-3478767953 .background-color-B6{background-color:#F7F8FE;}
		.d2-3478767953 .background-color-AA2{background-color:#4A6FF3;}
		.d2-3478767953 .background-color-AA4{background-color:#EDF0FD;}
		.d2-3478767953 .background-color-AA5{background-color:#F7F8FE;}
		.d2-3478767953 .background-color-AB4{background-color:#EDF0FD;}
		.d2-3478767953 .background-color-AB5{background-color:#F7F8FE;}
		.d2-3478767953 .color-N1{color:#0A0F25;}
		.d2-3478767953 .color-N2{color:#676C7E;}
		.d2-3478767953 .color-N3{color:#9499AB;}
		.d2-3478767953 .color-N4{color:#CFD2DD;}
		.d2-3478767953 .color-N5{color:#DEE1EB;}
		.d2-3478767953 .color-N6{color:#EEF1F8;}
		.d2-3478767953 .color-N7{color:#FFFFFF;}
		.d2-3478767953 .color-B1{color:#0D32B2;}
		.d2-3478767953 .color-B2{color:#0D32B2;}
		.d2-3478767953 .color-B3{color:#E3E9FD;}
		.d2-3478767953 .color-B4{color:#E3E9FD;}
		.d2-3478767953 .color-B5{color:#EDF0FD;}
		.d2-3478767953 .color-B6{color:#F7F8FE;}
		.d2-3478767953 .color-AA2{color:#4A6FF3;}
		.d2-3478767953 .color-AA4{color:#EDF0FD;}
		.d2-3478767953 .color-AA5{color:#F7F8FE;}
		.d2-3478767953 .color-AB4{color:#EDF0FD;}
		.d2-3478767953 .color-AB5{color:#F7F8FE;}.appendix text.text{fill:#0A0F25}.md{--color-fg-default:#0A0F25;--color-fg-muted:#676C7E;--color-fg-subtle:#9499AB;--color-canvas-default:#FFFFFF;--color-canvas-subtle:#EEF1F8;--color-border-default:#0D32B2;--color-border-muted:#0D32B2;--color-neutral-muted:#EEF1F8;--color-accent-fg:#0D32B2;--color-accent-emphasis:#0D32B2;--color-attention-subtle:#676C7E;--color-danger-fg:red;}.sketch-overlay-B1{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-B2{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-B3{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-B4{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-B5{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-B6{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-AA2{fill:url(#streaks-dark-d2-3478767953);mix-blend-mode:overlay}.sketch-overlay-AA4{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-AA5{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-AB4{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-AB5{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-N1{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-N2{fill:url(#streaks-dark-d2-3478767953);mix-blend-mode:overlay}.sketch-overlay-N3{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-N4{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-N5{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-N6{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.sketch-overlay-N7{fill:url(#streaks-bright-d2-3478767953);mix-blend-mode:darken}.light-code{display: block}.dark-code{display: none}@media screen and (prefers-color-scheme:dark){
		.d2-3478767953 .fill-N1{fill:#CDD6F4;}
		.d2-3478767953 .fill-N2{fill:#BAC2DE;}
		.d2-3478767953 .fill-N3{fill:#A6ADC8;}
		.d2-3478767953 .fill-N4{fill:#585B70;}
		.d2-3478767953 .fill-N5{fill:#45475A;}
		.d2-3478767953 .fill-N6{fill:#313244;}
		.d2-3478767953 .fill-N7{fill:#1E1E2E;}
		.d2-3478767953 .fill-B1{fill:#CBA6f7;}
		.d2-3478767953 .fill-B2{fill:#CBA6f7;}
		.d2-3478767953 .fill-B3{fill:#6C7086;}
		.d2-3478767953 .fill-B4{fill:#585B70;}
		.d2-3478767953 .fill-B5{fill:#45475A;}
		.d2-3478767953 .fill-B6{fill:#313244;}
		.d2-3478767953 .fill-AA2{fill:#f38BA8;}
		.d2-3478767953 .fill-AA4{fill:#45475A;}
		.d2-3478767953 .fill-AA5{fill:#313244;}
		.d2-3478767953 .fill-AB4{fill:#45475A;}
		.d2-3478767953 .fill-AB5{fill:#313244;}
		.d2-3478767953 .stroke-N1{stroke:#CDD6F4;}
		.d2-3478767953 .stroke-N2{stroke:#BAC2DE;}
		.d2-3478767953 .stroke-N3{stroke:#A6ADC8;}
		.d2-3478767953 .stroke-N4{stroke:#585B70;}
		.d2-3478767953 .stroke-N5{stroke:#45475A;}
		.d2-3478767953 .stroke-N6{stroke:#313244;}
		.d2-3478767953 .stroke-N7{stroke:#1E1E2E;}
		.d2-3478767953 .stroke-B1{stroke:#CBA6f7;}
		.d2-3478767953 .stroke-B2{stroke:#CBA6f7;}
		.d2-3478767953 .stroke-B3{stroke:#6C7086;}
		.d2-3478767953 .stroke-B4{stroke:#585B70;}
		.d2-3478767953 .stroke-B5{stroke:#45475A;}
		.d2-3478767953 .stroke-B6{stroke:#313244;}
		.d2-3478767953 .stroke-AA2{stroke:#f38BA8;}
		.d2-3478767953 .stroke-AA4{stroke:#45475A;}
		.d2-3478767953 .stroke-AA5{stroke:#313244;}
		.d2-3478767953 .stroke-AB4{stroke:#45475A;}
		.d2-3478767953 .stroke-AB5{stroke:#313244;}
		.d2-3478767953 .background-color-N1{background-color:#CDD6F4;}
		.d2-3478767953 .background-color-N2{background-color:#BAC2DE;}
		.d2-3478767953 .background-color-N3{background-color:#A6ADC8;}
		.d2-3478767953 .background-color-N4{background-color:#585B70;}
		.d2-3478767953 .background-color-N5{background-color:#45475A;}
		.d2-3478767953 .background-color-N6{background-color:#313244;}
		.d2-3478767953 .background-color-N7{background-color:#1E1E2E;}
		.d2-3478767953 .background-color-B1{background-color:#CBA6f7;}
		.d2-3478767953 .background-color-B2{background-color:#CBA6f7;}
		.d2-3478767953 .background-color-B3{background-color:#6C7086;}
		.d2-3478767953 .background-color-B4{background-color:#585B70;}
		.d2-3478767953 .background-color-B5{background-color:#45475A;}
		.d2-3478767953 .background-color-B6{background-color:#313244;}
		.d2-3478767953 .background-color-AA2{background-color:#f38BA8;}
		.d2-3478767953 .background-color-AA4{background-color:#45475A;}
		.d2-3478767953 .background-color-AA5{background-color:#313244;}
		.d2-3478767953 .background-color-AB4{background-color:#45475A;}
		.d2-3478767953 .background-color-AB5{background-color:#313244;}
		.d2-3478767953 .color-N1{color:#CDD6F4;}
		.d2-3478767953 .color-N2{color:#BAC2DE;}
		.d2-3478767953 .color-N3{color:#A6ADC8;}
		.d2-3478767953 .color-N4{color:#585B70;}
		.d2-3478767953 .color-N5{color:#45475A;}
		.d2-3478767953 .color-N6{color:#313244;}
		.d2-3478767953 .color-N7{color:#1E1E2E;}
		.d2-3478767953 .color-B1{color:#CBA6f7;}
		.d2-3478767953 .color-B2{color:#CBA6f7;}
		.d2-3478767953 .color-B3{color:#6C7086;}
		.d2-3478767953 .color-B4{color:#585B70;}
		.d2-3478767953 .color-B5{color:#45475A;}
		.d2-3478767953 .color-B6{color:#313244;}
		.d2-3478767953 .color-AA2{color:#f38BA8;}
		.d2-3478767953 .color-AA4{color:#45475A;}
		.d2-3478767953 .color-AA5{color:#313244;}
		.d2-3478767953 .color-AB4{color:#45475A;}
		.d2-3478767953 .color-AB5{color:#313244;}.appendix text.text{fill:#CDD6F4}.md{--color-fg-default:#CDD6F4;--color-fg-muted:#BAC2DE;--color-fg-subtle:#A6ADC8;--color-canvas-default:#1E1E2E;--color-canvas-subtle:#313244;--color-border-default:#CBA6f7;--color-border-muted:#CBA6f7;--color-neutral-muted:#313244;--color-accent-fg:#CBA6f7;--color-accent-emphasis:#CBA6f7;--color-attention-subtle:#BAC2DE;--color-danger-fg:red;}.sketch-overlay-B1{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-B2{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-B3{fill:url(#streaks-dark-d2-3478767953);mix-blend-mode:overlay}.sketch-overlay-B4{fill:url(#streaks-dark-d2-3478767953);mix-blend-mode:overlay}.sketch-overlay-B5{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-B6{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-AA2{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-AA4{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-AA5{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-AB4{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-AB5{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-N1{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-N2{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-N3{fill:url(#streaks-normal-d2-3478767953);mix-blend-mode:color-burn}.sketch-overlay-N4{fill:url(#streaks-dark-d2-3478767953);mix-blend-mode:overlay}.sketch-overlay-N5{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-N6{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.sketch-overlay-N7{fill:url(#streaks-darker-d2-3478767953);mix-blend-mode:lighten}.light-code{display: none}.dark-code{display: block}}]]></style><g class="aW5wdXQ= data"><g class="shape" ><path d="M 569 36 C 569 12 682 12 695 12 C 708 12 821 12 821 36 V 106 C 821 130 708 130 695 130 C 682 130 569 130 569 106 V 36 Z" stroke="#3973A4" fill="#E8F1FB" style="stroke-width:2;" /><path d="M 569 36 C 569 60 682 60 695 60 C 708 60 821 60 821 36" stroke="#3973A4" fill="#E8F1FB" style="stroke-width:2;" /></g><text x="695.000000" y="88.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Immutable DEV feature cache</text></g><g class="aXNvbGF0aW9u process"><g class="shape" ><rect x="559.000000" y="200.000000" width="272.000000" height="66.000000" stroke="#64748B" fill="#F4F7FA" style="stroke-width:2;" /></g><text x="695.000000" y="238.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Outer temporal cross-validation</text></g><g class="cTE="><g class="shape" ><rect x="12.000000" y="442.000000" width="653.000000" height="480.000000" class=" stroke-B1 fill-B4" style="stroke-width:2;" /></g><text x="338.500000" y="475.000000" class="text fill-N1" style="text-anchor:middle;font-size:28px">Q1: outcome decoder</text></g><g class="c3RhdGU="><g class="shape" ><rect x="810.000000" y="442.000000" width="475.000000" height="763.000000" class=" stroke-B1 fill-B4" style="stroke-width:2;" /></g><text x="1047.500000" y="475.000000" class="text fill-N1" style="text-anchor:middle;font-size:28px">State probability model</text></g><g class="cTI="><g class="shape" ><rect x="532.000000" y="1386.000000" width="590.000000" height="815.000000" class=" stroke-B1 fill-B4" style="stroke-width:2;" /></g><text x="827.000000" y="1419.000000" class="text fill-N1" style="text-anchor:middle;font-size:28px">Q2: incremental prediction</text></g><g class="cTEuZmVhdHVyZXM= process"><g class="shape" ><rect x="228.000000" y="492.000000" width="233.000000" height="66.000000" stroke="#64748B" fill="#F4F7FA" style="stroke-width:2;" /></g><text x="344.500000" y="530.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">K = 50 selected neural cells</text></g><g class="cTEubG9naXN0aWM= model"><g class="shape" ><rect x="192.000000" y="628.000000" width="304.000000" height="82.000000" stroke="#27845A" fill="#E8F7EF" style="stroke-width:2;" /></g><text x="344.000000" y="666.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="344.000000" dy="0.000000">Class-balanced L2 logistic regression</tspan><tspan x="344.000000" dy="18.500000">C = 10⁻⁴</tspan></text></g><g class="cTEuYXVj evaluation"><g class="shape" ><path d="M 132 790 L 62 824 L 132 859 L 271 859 L 341 824 L 271 790 Z" stroke="#B7791F" fill="#FFF4DE" style="stroke-width:2;" /></g><text x="201.500000" y="830.000000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Pooled out-of-fold AUC</text></g><g class="cTEuaW50ZXJwcmV0YXRpb24= output"><g class="shape" ><rect x="361.000000" y="790.000000" width="254.000000" height="82.000000" stroke="#8056A6" fill="#F3ECFA" style="stroke-width:2;" /></g><text x="488.000000" y="828.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="488.000000" dy="0.000000">Cell selection frequency</tspan><tspan x="488.000000" dy="18.500000">and standardized coefficients</tspan></text></g><g class="c3RhdGUuZmVhdHVyZXM= process"><g class="shape" ><rect x="884.000000" y="492.000000" width="232.000000" height="66.000000" stroke="#64748B" fill="#F4F7FA" style="stroke-width:2;" /></g><text x="1000.000000" y="530.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Pre-change neural activity</text></g><g class="c3RhdGUubG9naXN0aWM= model"><g class="shape" ><rect x="903.000000" y="628.000000" width="194.000000" height="82.000000" stroke="#27845A" fill="#E8F7EF" style="stroke-width:2;" /></g><text x="1000.000000" y="666.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="1000.000000" dy="0.000000">Natural-prevalence</tspan><tspan x="1000.000000" dy="18.500000">L2 logistic regression</tspan></text></g><g class="c3RhdGUuaW5uZXI= process"><g class="shape" ><rect x="898.000000" y="780.000000" width="205.000000" height="66.000000" stroke="#64748B" fill="#F4F7FA" style="stroke-width:2;" /></g><text x="1000.500000" y="818.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Inner out-of-fold logits</text></g><g class="c3RhdGUuY2FsaWJyYXRvcg== model"><g class="shape" ><rect x="860.000000" y="916.000000" width="281.000000" height="66.000000" stroke="#27845A" fill="#E8F7EF" style="stroke-width:2;" /></g><text x="1000.500000" y="954.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Training-only sigmoid calibration</text></g><g class="c3RhdGUubWV0cmljcw== evaluation"><g class="shape" ><path d="M 1020 1062 L 949 1108 L 1020 1155 L 1163 1155 L 1234 1108 L 1163 1062 Z" stroke="#B7791F" fill="#FFF4DE" style="stroke-width:2;" /></g><text x="1091.500000" y="1106.000000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="1091.500000" dy="0.000000">Conditional AUC</tspan><tspan x="1091.500000" dy="18.500000">calibrated log-loss gain</tspan></text></g><g class="cTIuY292YXJpYXRlcw== process"><g class="shape" ><rect x="582.000000" y="1444.000000" width="226.000000" height="66.000000" stroke="#64748B" fill="#F4F7FA" style="stroke-width:2;" /></g><text x="695.000000" y="1482.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">M0: behavioral covariates</text></g><g class="cTIubmV1cmFs process"><g class="shape" ><rect x="828.000000" y="1436.000000" width="244.000000" height="82.000000" stroke="#64748B" fill="#F4F7FA" style="stroke-width:2;" /></g><text x="950.000000" y="1474.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="950.000000" dy="0.000000">M1: covariates + cross-fitted</tspan><tspan x="950.000000" dy="18.500000">neural score</tspan></text></g><g class="cTIuc2VsZWN0aW9u process"><g class="shape" ><rect x="709.000000" y="1598.000000" width="226.000000" height="82.000000" stroke="#64748B" fill="#F4F7FA" style="stroke-width:2;" /></g><text x="822.000000" y="1636.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="822.000000" dy="0.000000">Nested one-SE C selection</tspan><tspan x="822.000000" dy="18.500000">10⁻⁴ … 10²</tspan></text></g><g class="cTIubTA= model"><g class="shape" ><rect x="594.000000" y="1760.000000" width="218.000000" height="66.000000" stroke="#27845A" fill="#E8F7EF" style="stroke-width:2;" /></g><text x="703.000000" y="1798.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Calibrated L2 logistic M0</text></g><g class="cTIubTE= model"><g class="shape" ><rect x="832.000000" y="1760.000000" width="218.000000" height="66.000000" stroke="#27845A" fill="#E8F7EF" style="stroke-width:2;" /></g><text x="941.000000" y="1798.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px">Calibrated L2 logistic M1</text></g><g class="cTIuZGVsdGE= evaluation"><g class="shape" ><path d="M 742 1906 L 661 1952 L 742 1999 L 903 1999 L 984 1952 L 903 1906 Z" stroke="#B7791F" fill="#FFF4DE" style="stroke-width:2;" /></g><text x="822.500000" y="1950.000000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="822.500000" dy="0.000000">Primary: Δ log-loss =</tspan><tspan x="822.500000" dy="18.500000">log-loss(M0) − log-loss(M1)</tspan></text></g><g class="cTIuc2Vjb25kYXJ5 output"><g class="shape" ><rect x="705.000000" y="2069.000000" width="234.000000" height="82.000000" stroke="#8056A6" fill="#F3ECFA" style="stroke-width:2;" /></g><text x="822.000000" y="2107.500000" class="text-bold fill-N1" style="text-anchor:middle;font-size:16px"><tspan x="822.000000" dy="0.000000">Secondary: Δ Brier, Δ AUC,</tspan><tspan x="822.000000" dy="18.500000">calibration, orthogonality</tspan></text></g><g class="cTEuKGZlYXR1cmVzIC0mZ3Q7IGxvZ2lzdGljKVswXQ=="><marker id="mk-d2-3478767953-3488378134" markerWidth="10.000000" markerHeight="12.000000" refX="7.000000" refY="6.000000" viewBox="0.000000 0.000000 10.000000 12.000000" orient="auto" markerUnits="userSpaceOnUse"> <polygon points="0.000000,0.000000 10.000000,6.000000 0.000000,12.000000" class="connection fill-B1" stroke-width="2" /> </marker><path d="M 344.750000 560.000000 L 344.750000 624.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTEuKGxvZ2lzdGljIC0mZ3Q7IGF1YylbMF0="><path d="M 294.081183 711.999999 L 294.003649 797.000002" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTEuKGxvZ2lzdGljIC0mZ3Q7IGludGVycHJldGF0aW9uKVswXQ=="><path d="M 395.415985 712.000000 L 395.415985 786.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="c3RhdGUuKGZlYXR1cmVzIC0mZ3Q7IGxvZ2lzdGljKVswXQ=="><path d="M 1000.500000 560.000000 L 1000.500000 624.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="c3RhdGUuKGxvZ2lzdGljIC0mZ3Q7IGlubmVyKVswXQ=="><path d="M 1000.500000 712.000000 L 1000.500000 776.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="c3RhdGUuKGlubmVyIC0mZ3Q7IGNhbGlicmF0b3IpWzBd"><path d="M 1000.500000 848.000000 L 1000.500000 912.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="c3RhdGUuKGNhbGlicmF0b3IgLSZndDsgbWV0cmljcylbMF0="><path d="M 1092.243750 983.999990 L 1092.012500 1058.000020" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTIuKGNvdmFyaWF0ZXMgLSZndDsgc2VsZWN0aW9uKVswXQ=="><path d="M 785.208008 1512.000000 L 785.208008 1594.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTIuKG5ldXJhbCAtJmd0OyBzZWxlY3Rpb24pWzBd"><path d="M 860.541016 1520.000000 L 860.541016 1594.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTIuKHNlbGVjdGlvbiAtJmd0OyBtMClbMF0="><path d="M 785.208008 1682.000000 L 785.208008 1756.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTIuKHNlbGVjdGlvbiAtJmd0OyBtMSlbMF0="><path d="M 860.541016 1682.000000 L 860.541016 1756.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTIuKG0wIC0mZ3Q7IGRlbHRhKVswXQ=="><path d="M 769.039990 1828.000000 L 769.002051 1902.000001" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTIuKG0xIC0mZ3Q7IGRlbHRhKVswXQ=="><path d="M 876.715308 1827.999987 L 876.985400 1902.000027" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="cTIuKGRlbHRhIC0mZ3Q7IHNlY29uZGFyeSlbMF0="><path d="M 823.000000 2001.000000 L 823.000000 2065.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="KGlucHV0IC0mZ3Q7IGlzb2xhdGlvbilbMF0="><path d="M 695.000000 132.000000 L 695.000000 196.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /></g><g class="KGlzb2xhdGlvbiAtJmd0OyBxMS5mZWF0dXJlcylbMF0="><path d="M 627.375000 268.000000 L 627.375000 296.000000 S 627.375000 306.000000 617.375000 306.000000 L 354.750000 306.000000 S 344.750000 306.000000 344.750000 316.000000 L 344.750000 488.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /><text x="413.500000" y="312.000000" class="text-italic fill-N2" style="text-anchor:middle;font-size:16px">late hit vs miss</text></g><g class="KGlzb2xhdGlvbiAtJmd0OyBzdGF0ZS5mZWF0dXJlcylbMF0="><path d="M 763.375000 268.000000 L 763.375000 296.000000 S 763.375000 306.000000 773.375000 306.000000 L 990.500000 306.000000 S 1000.500000 306.000000 1000.500000 316.000000 L 1000.500000 488.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /><text x="955.000000" y="312.000000" class="text-italic fill-N2" style="text-anchor:middle;font-size:16px">engaged vs disengaged</text></g><g class="KGlzb2xhdGlvbiAtJmd0OyBxMi5jb3ZhcmlhdGVzKVswXQ=="><path d="M 737.000000 268.000000 L 737.000000 1440.000000" fill="none" class="connection stroke-B1" style="stroke-width:2;" marker-end="url(#mk-d2-3478767953-3488378134)" mask="url(#d2-3478767953)" /><text x="737.000000" y="861.000000" class="text-italic fill-N2" style="text-anchor:middle;font-size:16px">outer-train only</text></g><g class="KHN0YXRlLmNhbGlicmF0b3IgLSZndDsgcTIubmV1cmFsKVswXQ=="><marker id="mk-d2-3478767953-2177206569" markerWidth="10.000000" markerHeight="12.000000" refX="7.000000" refY="6.000000" viewBox="0.000000 0.000000 10.000000 12.000000" orient="auto" markerUnits="userSpaceOnUse"> <polygon points="0.000000,0.000000 10.000000,6.000000 0.000000,12.000000" class="connection fill-B2" stroke-width="2" /> </marker><path d="M 908.750000 984.000000 L 908.750000 1432.000000" fill="none" class="connection stroke-B2" style="stroke-width:2;stroke-dasharray:8.000000,7.892511;" marker-end="url(#mk-d2-3478767953-2177206569)" mask="url(#d2-3478767953)" /><text x="909.000000" y="1215.000000" class="text-italic fill-N2" style="text-anchor:middle;font-size:16px">cross-fitted state score</text></g><mask id="d2-3478767953" maskUnits="userSpaceOnUse" x="-13" y="-13" width="1323" height="2239">
<rect x="-13" y="-13" width="1323" height="2239" fill="white"></rect>
<rect x="363.000000" y="296.000000" width="101" height="21" fill="black"></rect>
<rect x="875.000000" y="296.000000" width="160" height="21" fill="black"></rect>
<rect x="683.000000" y="845.000000" width="108" height="21" fill="black"></rect>
<rect x="833.000000" y="1199.000000" width="152" height="21" fill="black"></rect>
</mask></svg></svg>


## 6. Static single-experiment analysis


In [ ]:
FEATURE_LABELS = {
    "events_baselined_post": "Events · baseline-subtracted · [0, 0.30)s",
    "events_unbaselined_pre": "Events · unbaselined · [-1, 0)s",
    "events_unbaselined_post": "Events · unbaselined · [0, 0.30)s",
    "events_baselined_full_pre": "Events · baseline-subtracted · full pre",
    "dff_baselined_post": "dF/F · baseline-subtracted · [0, 0.30)s",
}

# Edit this compact configuration block, then rerun the cell.
SELECTED_EXPERIMENT_ID = int(index.iloc[0].ophys_experiment_id)
REPRESENTATION = "events_baselined_post"
HEATMAP_SCALE = "z"  # "z" or "raw"
CELL_ORDER = "effect"  # "effect" or "variance"
PCA_COLOR = "outcome"  # "outcome", "engaged_B", or "session_position"

DECODER_REPRESENTATION = "events_baselined_post"
DECODER_K = 50
DECODER_C = 1e-4
DECODER_SEEDS = 10
DECODER_CV = "blocked"  # "blocked" registered; "random" exploratory

if SELECTED_EXPERIMENT_ID not in set(index.ophys_experiment_id.astype(int)):
    raise ValueError(
        f"SELECTED_EXPERIMENT_ID={SELECTED_EXPERIMENT_ID} is not in the DEV cache."
    )
if REPRESENTATION not in FEATURE_LABELS:
    raise ValueError(f"Unknown REPRESENTATION: {REPRESENTATION}")


def selected_effect(values, labels):
    late = labels.late_hit.fillna(False).to_numpy(bool)
    miss = labels.miss.fillna(False).to_numpy(bool)
    engaged = labels.engaged_B.fillna(False).to_numpy(bool)
    keep = labels.keep_B.fillna(False).to_numpy(bool)
    left = values[late & engaged & keep]
    right = values[miss & engaged & keep]
    if not len(left) or not len(right):
        return np.zeros(values.shape[1])
    pooled = np.sqrt((np.nanvar(left, axis=0) + np.nanvar(right, axis=0)) / 2)
    return np.divide(
        np.nanmean(left, axis=0) - np.nanmean(right, axis=0),
        pooled,
        out=np.zeros(values.shape[1]),
        where=np.isfinite(pooled) & (pooled > 0),
    )


def bootstrap_population(frame, groups, n_boot=500):
    rng = np.random.default_rng(20260717)
    rows = []
    for label in ["late hit", "miss", "early hit", "abort", "other"]:
        values = frame[np.asarray(groups) == label]
        if not len(values):
            continue
        means = np.array([
            rng.choice(values, len(values), replace=True).mean() for _ in range(n_boot)
        ])
        rows.append({
            "outcome": label,
            "mean": float(np.mean(values)),
            "low": float(np.quantile(means, 0.025)),
            "high": float(np.quantile(means, 0.975)),
            "n": len(values),
        })
    return pd.DataFrame(rows)


def render_matrix():
    oeid = SELECTED_EXPERIMENT_ID
    matrix, labels, _ = experiment_data(oeid, REPRESENTATION)
    values = matrix.values.astype(float)
    effect = selected_effect(values, labels)
    order = (
        np.argsort(np.abs(effect))[::-1]
        if CELL_ORDER == "effect"
        else np.argsort(np.nanvar(values, axis=0))[::-1]
    )
    order = order[: min(200, len(order))]
    trial_outcome = outcome_labels(labels)
    trial_order = np.lexsort((labels.trial_index.to_numpy(), trial_outcome))
    shown = values[trial_order][:, order]
    if HEATMAP_SCALE == "z":
        mean = np.nanmean(shown, axis=0)
        sd = np.nanstd(shown, axis=0)
        shown = np.divide(shown - mean, sd, out=np.zeros_like(shown), where=sd > 0)
    population = np.nanmean(values, axis=1)
    summary = bootstrap_population(population, trial_outcome)
    with nullcontext():
        display(HTML(
            f"<h3>Experiment {oeid}</h3><p>{FEATURE_LABELS[REPRESENTATION]} · "
            f"{len(values):,} trials × {values.shape[1]:,} cells. "
            f"Heatmap shows the top {len(order)} cells.</p>"
        ))
        heat_frame = pd.DataFrame(
            shown,
            index=labels.trial_index.to_numpy()[trial_order],
            columns=[str(int(matrix.cell_ids[i])) for i in order],
        )
        figure, axis = plt.subplots(figsize=(15, 8))
        sns.heatmap(
            heat_frame,
            cmap="vlag",
            center=0,
            xticklabels=max(1, len(order) // 20),
            yticklabels=max(1, len(heat_frame) // 25),
            cbar_kws={
                "label": "Cell z-score" if HEATMAP_SCALE == "z" else "Response"
            },
            ax=axis,
        )
        axis.set(
            title="Trial × cell response matrix",
            xlabel="Cell specimen ID",
            ylabel="Raw trial index (grouped by outcome)",
        )
        display_matplotlib_figure(figure)

        figure, axis = plt.subplots(figsize=(9, 5))
        lower = summary["mean"] - summary.low
        upper = summary.high - summary["mean"]
        axis.bar(
            summary.outcome,
            summary["mean"],
            yerr=np.vstack([lower, upper]),
            color=sns.color_palette("deep", len(summary)),
            capsize=4,
        )
        for position, row in enumerate(summary.itertuples()):
            axis.annotate(
                f"n={row.n}",
                (position, row.high),
                xytext=(0, 6),
                textcoords="offset points",
                ha="center",
            )
        axis.set(
            title="Population response by outcome (bootstrap 95% CI)",
            xlabel="Outcome",
            ylabel="Mean across cells",
        )
        display_matplotlib_figure(figure)

        effect_table = pd.DataFrame({
            "cell_id": matrix.cell_ids,
            "effect": effect,
            "variance": np.nanvar(values, axis=0),
        }).sort_values("effect")
        effect_table["rank"] = np.arange(len(effect_table))
        figure, axis = plt.subplots(figsize=(11, 5))
        points = axis.scatter(
            effect_table["rank"],
            effect_table["effect"],
            c=effect_table["variance"],
            cmap="viridis",
            s=28,
            alpha=0.8,
        )
        figure.colorbar(points, ax=axis, label="Variance")
        axis.axhline(0, color="0.45", linewidth=1)
        axis.set(
            title="Per-cell standardized late-hit − miss effect",
            xlabel="Effect rank",
            ylabel="Standardized effect",
        )
        display_matplotlib_figure(figure)

        distribution = pd.DataFrame({
            "population_response": population,
            "outcome": trial_outcome,
        })
        figure, axis = plt.subplots(figsize=(9, 5))
        sns.violinplot(
            data=distribution,
            x="outcome",
            y="population_response",
            hue="outcome",
            inner="box",
            cut=0,
            legend=False,
            ax=axis,
        )
        axis.set(
            title="Trial-level population-response distributions",
            xlabel="Outcome",
            ylabel="Population response",
        )
        display_matplotlib_figure(figure)

        comparisons = [
            ("events_baselined_post", "dff_baselined_post"),
            ("events_unbaselined_pre", "events_unbaselined_post"),
        ]
        titles = ["Events post vs dF/F post", "Events pre vs events post"]
        figure, axes = plt.subplots(1, 2, figsize=(13, 5))
        for axis, title, (left_name, right_name) in zip(axes, titles, comparisons):
            left = experiment_data(oeid, left_name)[0].values.mean(axis=1)
            right = experiment_data(oeid, right_name)[0].values.mean(axis=1)
            correlation = np.corrcoef(left, right)[0, 1]
            sns.scatterplot(x=left, y=right, s=25, alpha=0.55, ax=axis)
            axis.set(
                title=f"{title} · r={correlation:.2f}",
                xlabel="Left population response",
                ylabel="Right population response",
            )
        figure.suptitle("Representation comparisons")
        display_matplotlib_figure(figure)


def render_geometry():
    oeid = SELECTED_EXPERIMENT_ID
    matrix, labels, q2 = experiment_data(oeid, REPRESENTATION)
    values = matrix.values.astype(float)
    finite = np.isfinite(values).all(axis=1)
    if finite.sum() < 3:
        print("PCA nonestimable: fewer than three finite trials.")
        return
    scaled = StandardScaler().fit_transform(values[finite])
    n_components = min(10, scaled.shape[0], scaled.shape[1])
    pca = PCA(n_components=n_components).fit(scaled)
    scores = pca.transform(scaled)
    plot = pd.DataFrame({"PC1": scores[:, 0], "PC2": scores[:, 1]})
    plot["outcome"] = outcome_labels(labels)[finite]
    plot["engaged_B"] = labels.engaged_B.fillna(False).to_numpy()[finite].astype(str)
    plot["session_position"] = q2.session_position.to_numpy()[finite]
    population = np.nanmean(values, axis=1)
    joint = q2.copy()
    joint["population_response"] = population
    joint["outcome"] = outcome_labels(labels)
    with nullcontext():
        figure, axis = plt.subplots(figsize=(9, 6))
        color_column = PCA_COLOR
        sns.scatterplot(
            data=plot,
            x="PC1",
            y="PC2",
            hue=color_column,
            palette="viridis" if color_column == "session_position" else "deep",
            s=42,
            alpha=0.75,
            ax=axis,
        )
        axis.set_title(f"Trial geometry · {FEATURE_LABELS[REPRESENTATION]}")
        axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        display_matplotlib_figure(figure)

        figure, axis = plt.subplots(figsize=(9, 5))
        sns.barplot(
            x=np.arange(1, n_components + 1),
            y=pca.explained_variance_ratio_,
            color=sns.color_palette()[0],
            ax=axis,
        )
        axis.set(
            title="PCA scree plot",
            xlabel="Principal component",
            ylabel="Explained variance ratio",
        )
        display_matplotlib_figure(figure)

        covariates = [
            ("pre_change_pupil", "Pre-change pupil"),
            ("pre_change_running", "Pre-change running"),
            ("session_position", "Session position"),
        ]
        figure, axes = plt.subplots(1, 3, figsize=(15, 5))
        for axis, (name, label_text) in zip(axes, covariates):
            good = joint[name].notna() & joint.population_response.notna()
            sns.regplot(
                data=joint.loc[good],
                x=name,
                y="population_response",
                scatter_kws={"s": 18, "alpha": 0.45},
                line_kws={"color": "#c44e52"},
                ax=axis,
            )
            axis.set_title(label_text)
        figure.suptitle("Behavior–neural relationships (no imputation)")
        display_matplotlib_figure(figure)

        missing = (
            q2.isna().mean().sort_values(ascending=False).rename("missing_fraction")
            .reset_index().rename(columns={"index": "covariate"})
        )
        figure_height = max(5, 0.3 * len(missing))
        figure, axis = plt.subplots(figsize=(10, figure_height))
        sns.barplot(
            data=missing,
            x="missing_fraction",
            y="covariate",
            color=sns.color_palette()[0],
            ax=axis,
        )
        axis.set(
            title="Q2 covariate missingness",
            xlabel="Missing fraction",
            ylabel="Covariate",
        )
        display_matplotlib_figure(figure)


def render_decoder():
    oeid = SELECTED_EXPERIMENT_ID
    feature = DECODER_REPRESENTATION
    k = DECODER_K
    C = DECODER_C
    n_seeds = DECODER_SEEDS
    cv = DECODER_CV
    config = DecoderConfig(k=k, C=C, n_seeds=n_seeds, cv=cv)
    label = "REGISTERED DEFAULTS" if not config.exploratory else "EXPLORATORY"
    color = "#166534" if not config.exploratory else "#b45309"
    display(HTML(
        f"<div style='padding:8px;border-left:4px solid {color}'><b>{label}</b> · "
        "Single-experiment educational analysis; not an authoritative mouse-level result."
        "</div>"
    ))
    print("Fitting train-only scalers and logistic models…")
    matrix, labels, _ = experiment_data(oeid, feature)
    result = run_q1_decoder(matrix, labels, config)
    with nullcontext():
        display(HTML(
            f"<div style='padding:8px;border-left:4px solid {color}'><b>{label}</b> · "
            "Single-experiment educational analysis; not an authoritative mouse-level result."
            "</div>"
        ))
        if result.status != "estimable":
            display(HTML(
                f"<h3>Decoder nonestimable</h3><code>{result.reason}</code>"
            ))
            return
        display(HTML(
            f"<h3>Mean seed AUC: {result.mean_auc:.3f}</h3>"
            f"<p>{len(result.oof):,} registered eligible trials · "
            f"{len(result.seed_metrics):,} deterministic seeds.</p>"
        ))
        figure, axis = plt.subplots(figsize=(9, 5))
        sns.barplot(
            data=result.seed_metrics,
            x="seed",
            y="auc",
            color=sns.color_palette()[0],
            ax=axis,
        )
        axis.axhline(0.5, linestyle="--", color="0.4")
        axis.set(
            title="Seed-level OOF AUC",
            xlabel="Seed",
            ylabel="OOF AUC",
            ylim=(0, 1),
        )
        display_matplotlib_figure(figure)

        fpr, tpr, _ = roc_curve(result.oof.y, result.oof.mean_score)
        figure, axis = plt.subplots(figsize=(7, 6))
        axis.plot(fpr, tpr, linewidth=2, label="Mean OOF score")
        axis.plot([0, 1], [0, 1], linestyle="--", color="0.5", label="Chance")
        axis.set(
            title="OOF ROC across mean seed scores",
            xlabel="False-positive rate",
            ylabel="True-positive rate",
            xlim=(0, 1),
            ylim=(0, 1),
        )
        axis.legend(frameon=False)
        display_matplotlib_figure(figure)

        fold = result.fold_metrics[
            result.fold_metrics.seed.eq(0)
        ].sort_values("fold")
        figure, axis = plt.subplots(figsize=(9, 5))
        axis.bar(
            fold.fold,
            fold.test_negative,
            label="miss",
            color=sns.color_palette("deep")[0],
        )
        axis.bar(
            fold.fold,
            fold.test_positive,
            bottom=fold.test_negative,
            label="late hit",
            color=sns.color_palette("deep")[1],
        )
        axis.set(
            title="Seed 0 temporal test-fold support",
            xlabel="Fold",
            ylabel="Trials",
        )
        axis.legend(frameon=False)
        display_matplotlib_figure(figure)

        temporal = result.oof.sort_values("trial_index").copy()
        temporal["outcome"] = temporal.y.map({0: "miss", 1: "late hit"})
        figure, axis = plt.subplots(figsize=(12, 5))
        sns.scatterplot(
            data=temporal,
            x="trial_index",
            y="mean_score",
            hue="outcome",
            style="outcome",
            s=32,
            alpha=0.75,
            ax=axis,
        )
        axis.axhline(0.5, linestyle="--", color="0.5")
        axis.set(
            title="OOF decision score over raw trial order",
            xlabel="Raw trial index",
            ylabel="Mean OOF score",
        )
        axis.legend(frameon=False)
        display_matplotlib_figure(figure)

        cells = result.cell_summary[
            result.cell_summary.selection_frequency.gt(0)
        ].sort_values(
            ["selection_frequency", "mean_abs_coefficient"], ascending=False
        ).head(60)
        figure, axis = plt.subplots(figsize=(10, 6))
        max_abs = max(float(cells.median_coefficient.abs().max()), 1e-12)
        sns.scatterplot(
            data=cells,
            x="selection_frequency",
            y="median_coefficient",
            size="mean_abs_coefficient",
            hue="median_coefficient",
            palette="vlag",
            hue_norm=(-max_abs, max_abs),
            sizes=(40, 320),
            alpha=0.8,
            ax=axis,
        )
        for row in cells.head(10).itertuples():
            axis.annotate(
                str(int(row.cell_id)),
                (row.selection_frequency, row.median_coefficient),
                xytext=(4, 4),
                textcoords="offset points",
                fontsize=8,
            )
        axis.axhline(0, color="0.45", linewidth=1)
        axis.set(
            title="Cell selection frequency and standardized coefficients",
            xlabel="Selection frequency",
            ylabel="Median standardized coefficient",
        )
        axis.legend(bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
        display_matplotlib_figure(figure)


selected_row = index.loc[
    index.ophys_experiment_id.astype(int).eq(SELECTED_EXPERIMENT_ID)
].iloc[0]
display(HTML(
    "<h2>Static single-experiment analysis</h2>"
    f"<p>Mouse {int(selected_row.mouse_id)} · "
    f"container {int(selected_row.ophys_container_id)} · "
    f"experiment {SELECTED_EXPERIMENT_ID} · "
    f"{selected_row.session_type}</p>"
    "<p>Edit the configuration block at the top of this cell and rerun it "
    "to inspect a different experiment or representation.</p>"
))

render_matrix()
render_geometry()
render_decoder()
